# Aquatic Guardian — Mentee 2 (Workstream D)
## Thermal–Tidal Transport, Ecological Dose, and Beneficial-Use Screening for Oceanic Data Centres

**Six-week Colab project. Free tier, open-source only, public data.**

---

### What this notebook is

A **quantitative screening system** for the question at the centre of Aquatic Guardian:

> *A coastal or submerged data centre rejects heat into tidal water. Where does that heat go,
> what does it do to the animals and plants that live there, is any of it recoverable for a
> contained aquaculture or agricultural use, and under what conditions must the facility stop?*

It is built as a chain of **vetted physical models**, not a demonstration. Every stage is
either derived from a conservation law or checked against an analytic solution, a published
reference value, or a hard physical bound — and the checks run in the notebook, so a
reviewer can see them fail if you break something.

```
   IT load  ──▶  heat rejection  ──▶  near-field dilution  ──▶  far-field tidal transport
   (MW, PUE)     (MW to water,        (jet/plume, bounded      (2-D advection–diffusion,
                  m³/s, ΔT)            by mass conservation)     validated solver)
                                                │
                                                ▼
        compliance  ◀──  ecological dose  ◀──  water-quality coupling
        (CCME, EPA R10,   (Lagrangian exposure,  (O₂ solubility, NH₃ speciation,
         WAC, OAR, CA)     TPC, aerobic scope,     respiration Q₁₀)
                │           thermal dose)
                ▼
    beneficial use (aquaculture / greenhouse)  +  tidally-aware curtailment schedule
                │
                ▼
        uncertainty (Monte Carlo + sensitivity)  ──▶  machine-readable safety rules
```

### What it is **not**

It is a **screening tool for scenario comparison and hypothesis generation**. It is not a
validated field prediction, it does not prove ecological safety, and no reuse or discharge
decision follows from it without human review, aquatic-science input, and later 3-D
hydrodynamic validation (Delft3D / MOHID / ROMS / CORMIX).

> **Standing rule from the project charter:** *no mentee acting alone decides that a
> thermal-reuse scenario is ecologically safe.* That judgment belongs to the graduate
> student and supervisor together with aquatic-science advice. This notebook produces
> **evidence that constrains** such a decision; it never makes one.

### How to run it

`Runtime → Run all`. It completes on a free CPU runtime. Then edit the `SITE` and `PLANT`
dictionaries in Part 1–2 and re-run. Export the artifacts in Part 12 to your GitHub repo.

---

### Six-week map

| Week | Parts | Deliverable |
|---|---|---|
| 1 | 0–2 | Site + regulatory envelope documented; plant heat balance closed |
| 2 | 3–5 | Near-field module; far-field solver **passing its verification suite** |
| 3 | 6–7 | Water-quality coupling; ecological dose–response with Lagrangian exposure |
| 4 | 8–9 | Compliance screening; beneficial-use (aquaculture + agriculture) assessment |
| 5 | 10–11 | Curtailment optimiser; Monte Carlo uncertainty + sensitivity ranking |
| 6 | 12 | Safety rule set, monitoring design, reproducible handoff |

## Part 0 — Environment, run configuration, and the safety contract

Everything below is stock Colab: `numpy`, `scipy`, `pandas`, `matplotlib`. No installs, no
GPU, no paid services. Optional real-data paths are guarded so the notebook always completes.

In [ ]:
# ---- library install (Colab vs local) ----------------------------------------
# Local / CI: editable install from the repo parent of notebooks/.
# Colab (optional Drive path for handoffs): /content/drive/MyDrive/aquatic_guardian/
import sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.check_call([
        sys.executable, "-m", "pip", "-q", "install",
        "git+https://github.com/ejri/aquatic-guardian.git@v0.1.0",
    ])
else:
    _repo = Path("..").resolve()
    subprocess.check_call([sys.executable, "-m", "pip", "-q", "install", "-e", str(_repo)])

import aquatic_guardian as ag
print("aquatic_guardian", ag.__version__)

HANDOFF_DIR = Path("_handoffs")
HANDOFF_DIR.mkdir(parents=True, exist_ok=True)
# Colab optional: HANDOFF_DIR = Path("/content/drive/MyDrive/aquatic_guardian")
print("handoff directory:", HANDOFF_DIR.resolve())


In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, json, textwrap, warnings, hashlib, time
from scipy.linalg import solve_banded
from scipy import stats

np.set_printoptions(suppress=True, precision=4)
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 40)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": .25,
                     "axes.spines.top": False, "axes.spines.right": False,
                     "font.size": 9, "figure.facecolor": "white"})

RUN = dict(
    SEED           = 20260219,
    USE_LIVE_DATA  = False,   # True in Colab to pull DFO/NOAA tide + ERDDAP records
    FAST           = True,    # False = finer grids + 2000 Monte Carlo draws (slower)
    EXPORT_DIR     = ".",
)
RNG = np.random.default_rng(RUN["SEED"])

SAFETY_CONTRACT = {
  "purpose": "screening estimates for scenario comparison and hypothesis generation",
  "not_for": ["field prediction", "proof of ecological safety", "regulatory certification",
              "autonomous control of cooling or discharge"],
  "human_review_required_for": ["any reuse decision", "any curtailment rule put into service",
                                "any claim that a scenario is ecologically acceptable"],
  "escalation": "graduate student + supervisor + aquatic-science advisor, jointly",
  "validation_gap": ["3-D hydrodynamics not run", "near-field mixing not measured",
                     "no biological response experiment", "thresholds are literature values, "
                     "not site-specific"],
}
print("Aquatic Guardian — thermal/tidal screening chain")
print(f"seed={RUN['SEED']}  live_data={RUN['USE_LIVE_DATA']}  fast={RUN['FAST']}")
print("\nSafety contract:")
for k, v in SAFETY_CONTRACT.items():
    print(f"  {k}: {v}")

### 0.1 Seawater properties — the foundation everything else stands on

Every downstream number depends on density $\rho$, specific heat $c_p$, and the thermal
expansion coefficient $\alpha$. We use **EOS-80** (UNESCO 1983) rather than fixed constants,
because $\alpha$ varies by a factor of three across the temperature range that matters here
(4–24 °C), and $\alpha$ sets the buoyancy of the discharge plume.

$$\alpha(T,S) = -\frac{1}{\rho}\frac{\partial\rho}{\partial T}$$

**Vetting:** the implementation is checked against the published UNESCO one-atmosphere
check values. If it does not reproduce them, the cell raises.

In [ ]:
# INTEGRATED: aquatic_guardian library
def rho_sw(T, S):
    """Seawater density via aquatic_guardian.seawater.density_eos80 (UNESCO EOS-80)."""
    T = np.asarray(T, float); S = np.asarray(S, float)
    if T.shape or S.shape:
        return np.vectorize(ag.seawater.density_eos80)(T, S)
    return ag.seawater.density_eos80(float(T), float(S))

def alpha_sw(T, S, h=0.01):
    """Thermal expansion coefficient, 1/K (central difference on EOS-80)."""
    return -(rho_sw(T+h, S) - rho_sw(T-h, S))/(2*h*rho_sw(T, S))

def cp_sw(T, S):
    """Specific heat at constant pressure, J/kg/K (Millero et al. 1973; Sharqawy et al. 2010)."""
    s = np.asarray(S, float); Tk = np.asarray(T, float) + 273.15
    A = 5.328 - 9.76e-2*s + 4.04e-4*s**2
    B = -6.913e-3 + 7.351e-4*s - 3.15e-6*s**2
    Cc = 9.6e-6 - 1.927e-6*s + 8.23e-9*s**2
    D = 2.5e-9 + 1.666e-9*s - 7.125e-12*s**2
    return 1000*(A + B*Tk + Cc*Tk**2 + D*Tk**3)

# ---- VET: UNESCO 1983 published check values --------------------------------
_CHECKS = [(0, 5, 999.96675), (0, 25, 997.04796), (35, 5, 1027.67547),
           (35, 25, 1023.34306), (35, 0, 1028.10623)]
_err = max(abs(float(rho_sw(T, S)) - r) for S, T, r in _CHECKS)
assert _err < 1e-3, f"EOS-80 implementation is wrong: max deviation {_err:.2e} kg/m3"
print(f"[VET] EOS-80 (library) reproduces UNESCO check values to {_err:.2e} kg/m3  -> PASS")

_tab = pd.DataFrame({"T_C": [4, 8, 12, 16, 20, 24]})
_tab["rho_S31"]   = [float(rho_sw(t, 31)) for t in _tab.T_C]
_tab["alpha_1perK"] = [float(alpha_sw(t, 31)) for t in _tab.T_C]
_tab["cp_J_kgK"]  = [float(cp_sw(t, 31)) for t in _tab.T_C]
_tab["buoyancy_of_+8K_plume_g'"] = [9.80665*float(alpha_sw(t, 31))*8 for t in _tab.T_C]
display(_tab.round(6))
print("Note: alpha triples from 4 C to 24 C, so the SAME +8 K discharge is three times more")
print("buoyant in summer. A model with constant alpha gets summer plume behaviour wrong.")


## Part 1 — Site, tidal regime, and the regulatory envelope

### 1.1 Why the regulatory numbers come first

For a West Coast site the thermal envelope is **already nearly exhausted by ambient summer
temperature**. The binding constraint is usually not "does the plume look big" but "does the
incremental warming fit inside a criterion of 0.3 °C or ±1 °C". So we encode the actual
criteria as data and test against them, rather than inventing placeholder thresholds.

| Jurisdiction | Instrument | Numeric criterion |
|---|---|---|
| Canada (federal) | CCME marine WQG (1999, upd. 2007) | ΔT ≤ **±1 °C** at any time/location/depth; rate ≤ **0.5 °C/h** |
| British Columbia | WQG for temperature, applied via EMA waste-discharge authorisation | same ±1 °C and 0.5 °C/h |
| Canada (federal) | *Fisheries Act* s. 34(1)(b), 35(1), 36(3) | heat is a "deleterious substance"; **no thermal-variance pathway** analogous to CWA 316(a) |
| US federal | EPA Region 10 guidance (910-B-03-002) | 7-DADM 13 °C spawning, 16 °C core rearing, 17.5–18 °C rearing/migration, 20 °C migration corridor; **0.3 °C** human-caused allowance |
| Washington | WAC 173-201A-200 | 12 °C char, 16 °C core salmonid, 17.5 °C rearing/migration; 0.3 °C cumulative; plume must not expose fish >2 s above 33 °C |
| Oregon | OAR 340-041-0028 | 12 °C bull trout, 13 °C spawning, 16 °C core cold-water, 18 °C rearing/migration; 0.3 °C allowance |
| California | Thermal Plan, Res. 75-89 | new discharge to enclosed bays: ΔT > **2.2 °C (4 °F)** prohibited; coastal ocean ΔT ≤ 4 °F beyond 1 000 ft |
| US federal | CWA §316(b), 40 CFR 125.90–99 | Best Technology Available for intakes withdrawing **> 2 MGD** |

Two consequences worth stating out loud before any modelling:

1. **The Canadian criterion is a ΔT criterion, not an absolute-temperature criterion.**
   It binds even in cold water. A facility can be well below any lethal temperature and
   still be non-compliant.
2. **The 0.5 °C/h rate-of-change criterion is almost never modelled**, because steady-state
   plume models cannot produce it. A tidal model can — and we compute it in Part 8.

In [ ]:
# INTEGRATED: aquatic_guardian library
CRITERIA = pd.DataFrame([
 # id,                jurisdiction, kind,        value, unit,   basis
 ("CCME_dT",          "Canada",     "delta_T",    1.0,  "K",    "CCME marine WQG: +/-1 C any time/location/depth"),
 ("CCME_rate",        "Canada",     "dT_dt",      0.5,  "K/h",  "CCME marine WQG: max rate of human-induced change"),
 ("BC_dT",            "BC",         "delta_T",    1.0,  "K",    "BC WQG temperature (marine/estuarine), via EMA permit"),
 ("EPA_R10_incr",     "US-fed",     "delta_T",    0.3,  "K",    "EPA R10 cold-water protection increment"),
 ("WA_incr",          "WA",         "delta_T",    0.3,  "K",    "WAC 173-201A-200 cumulative human-caused"),
 ("OR_incr",          "OR",         "delta_T",    0.3,  "K",    "OAR 340-041-0028 human-use allowance"),
 ("CA_bay_dT",        "CA",         "delta_T",    2.2,  "K",    "CA Thermal Plan 75-89, new discharge, enclosed bay"),
 ("EPA_R10_spawn",    "US-fed",     "abs_7DADM", 13.0,  "C",    "salmonid spawning / egg incubation"),
 ("EPA_R10_core",     "US-fed",     "abs_7DADM", 16.0,  "C",    "core juvenile rearing (cold water)"),
 ("EPA_R10_rear",     "US-fed",     "abs_7DADM", 17.5, "C",    "salmon & steelhead rearing and migration"),
 ("EPA_R10_migr",     "US-fed",     "abs_7DADM", 20.0,  "C",    "migration corridor"),
 ("WA_lethal_7DADM",  "WA",         "abs_7DADM", 22.0,  "C",    "lethal-plume threshold"),
 ("CWA_316b_MGD",     "US-fed",     "intake",     2.0,  "MGD",  "BTA applies above this withdrawal"),
], columns=["id", "jurisdiction", "kind", "value", "unit", "basis"])

print("Regulatory envelope encoded as data (edit for your site's designated uses):")
display(CRITERIA)

BINDING_SET = ["CCME_dT", "CCME_rate", "EPA_R10_incr", "EPA_R10_core", "EPA_R10_rear"]
print(f"\nDefault binding set for a BC / Salish Sea screening: {BINDING_SET}")
print("The tightest DELTA-T criterion in this set is "
      f"{CRITERIA.set_index('id').loc[[c for c in BINDING_SET if CRITERIA.set_index('id').loc[c,'kind']=='delta_T'],'value'].min()} K.")

# Library criteria catalogue (thresholds with provenance in data/criteria.json)
print("\n[library] criteria.json entries:", len(ag.criteria.load_criteria()))
print("[library] WA / CA temperature_rule demonstrated in the compliance cell (Part 9).")


### 1.2 Tidal regime — harmonic synthesis with a live-data hook

The tide sets the flushing timescale, the excursion length, and therefore the recirculation
risk. The **form factor**

$$F = \frac{A_{K_1} + A_{O_1}}{A_{M_2} + A_{S_2}}$$

classifies the regime: $F<0.25$ semidiurnal, $0.25$–$1.5$ mixed mainly semidiurnal
(the BC coast), $1.5$–$3$ mixed mainly diurnal, $>3$ diurnal.

The **tidal excursion** $L_{ex} = U_0 T/\pi$ is the peak-to-peak distance a water parcel
travels. It is the single most decision-relevant length in this notebook: *an intake inside
$L_{ex}$ of the outfall re-entrains its own discharge.*

Live retrieval from DFO/CHS (Canada) or NOAA CO-OPS (US) is coded below and **guarded**;
with `USE_LIVE_DATA=False` we synthesise from documented constituent amplitudes instead.

### Library note — tidal excursion (Correction 2)

The library reports the **half-cycle** excursion `2U/ω` for a single peak constituent (`ag.tidal.excursion_half_cycle`). Over whole cycles the oscillation cancels and only residual drift remains (`ag.tidal.residual_drift`). See [docs/CORRECTIONS.md](../docs/CORRECTIONS.md) item 2. Numbers below that used a multi-constituent `U_eff` will differ.

In [ ]:
# INTEGRATED: aquatic_guardian library
CONSTITUENTS = {"M2": 12.4206012, "S2": 12.0, "N2": 12.6583482,
                "K1": 23.9344696, "O1": 25.8193417, "P1": 24.0658902}   # periods, hours

SITE = dict(
    name       = "SET_ME — e.g. Strait of Georgia nearshore, BC",
    station    = "SET_ME — DFO/CHS station id or NOAA CO-OPS id",
    source_url = "https://www.tides.gc.ca/   |   https://tidesandcurrents.noaa.gov/",
    retrieved  = "SET_ME — retrieval date",
    # tidal CURRENT amplitudes (m/s) along the dominant axis, not water-level amplitudes
    U_amps     = {"M2": 0.30, "S2": 0.09, "N2": 0.06, "K1": 0.15, "O1": 0.09},
    U_phases   = {"M2": 0.0,  "S2": 25.0, "N2": -18.0, "K1": 70.0, "O1": 55.0},
    depth_H    = 14.0,     # mean depth at the outfall, m
    T_amb_mean = 11.5,     # annual mean ambient, C
    T_amb_summer_7DADM = 16.8,   # summer 7-day avg daily max, C  <-- the number that matters
    S_amb      = 30.5,     # PSU
    pH_amb     = 7.95,
    K_long     = 12.0,     # longitudinal dispersion, m2/s  (literature 1-50 nearshore)
    K_lat      = 3.0,      # lateral dispersion, m2/s
    axis_deg   = 0.0,      # along-channel axis of the dominant current
)

def tidal_current(t_s, amps, phases=None):
    phases = phases or {}
    t = np.asarray(t_s, float); u = np.zeros_like(t)
    for k, a in amps.items():
        u = u + a*np.sin(2*np.pi*t/(CONSTITUENTS[k]*3600.0) + np.deg2rad(phases.get(k, 0.0)))
    return u

def form_factor(a):
    return (a.get("K1",0)+a.get("O1",0))/max(a.get("M2",0)+a.get("S2",0), 1e-12)

def classify_regime(F):
    return ("semidiurnal" if F < 0.25 else "mixed, mainly semidiurnal" if F < 1.5
            else "mixed, mainly diurnal" if F < 3.0 else "diurnal")

def fetch_tides_live(station, days=30):
    '''Optional live path. Kept guarded so the notebook never depends on the network.'''
    import urllib.request
    url = ("https://api.tidesandcurrents.noaa.gov/api/prod/datagetter?"
           f"product=currents&station={station}&date=recent&units=metric&"
           "time_zone=gmt&format=json")
    with urllib.request.urlopen(url, timeout=20) as r:
        return json.loads(r.read().decode())

if RUN["USE_LIVE_DATA"]:
    try:
        raw = fetch_tides_live(SITE["station"])
        print("Live record retrieved. Fit constituents with `utide` (pip install utide) and "
              "replace SITE['U_amps']/['U_phases'] with the fitted values.")
    except Exception as e:
        print(f"Live retrieval failed ({type(e).__name__}); falling back to synthesis.")

F = form_factor(SITE["U_amps"])
T_M2 = CONSTITUENTS["M2"]*3600.0
U_eff = np.sqrt(sum(a**2 for a in SITE["U_amps"].values())/2)*np.sqrt(2)   # amplitude-equivalent
# Library (Correction 2): half-cycle excursion for the M2 peak constituent.
# Older notebook used an amplitude-equivalent U_eff over all constituents (U_eff*T_M2/pi).
L_ex_legacy = U_eff*T_M2/np.pi
L_ex = ag.tidal.excursion_half_cycle(SITE["U_amps"]["M2"])
L_res_2cyc = ag.tidal.residual_drift(0.02, n_cycles=2)  # example residual 2 cm/s over 2 cycles
print(f"[library] M2 half-cycle excursion = {L_ex:,.0f} m "
      f"(legacy U_eff formula gave {L_ex_legacy:,.0f} m); "
      f"residual drift (2 cm/s, 2 cycles) = {L_res_2cyc:,.0f} m")


t = np.arange(0, 15*86400, 300.)
u = tidal_current(t, SITE["U_amps"], SITE["U_phases"])
fig, ax = plt.subplots(2, 1, figsize=(11, 4.6), height_ratios=[2, 1])
ax[0].plot(t/86400, u, lw=.8, color="#1F3A5F")
ax[0].axhline(0, color="k", lw=.5)
ax[0].set_ylabel("along-axis current (m/s)")
ax[0].set_title(f"Synthesised tidal current | F = {F:.2f} -> {classify_regime(F)} | "
                f"L_ex = {L_ex:,.0f} m")
ax[1].plot(t[:int(3*86400/300)]/3600, u[:int(3*86400/300)], lw=1.1, color="#2E7D82")
ax[1].axhline(0, color="k", lw=.5); ax[1].set_xlabel("hours"); ax[1].set_ylabel("m/s")
ax[1].set_title("first three days — note the unequal successive maxima (mixed regime)", fontsize=8)
plt.tight_layout(); plt.show()

print(f"form factor F        = {F:.3f}  -> {classify_regime(F)}")
print(f"rms current          = {np.sqrt(np.mean(u**2)):.3f} m/s")
print(f"amplitude-equivalent = {U_eff:.3f} m/s")
print(f"tidal excursion L_ex = {L_ex:,.0f} m   <-- any intake inside this re-entrains discharge")
print(f"slack fraction (|u| < 0.05 m/s) = {100*np.mean(np.abs(u) < 0.05):.1f}% of the time")


## Part 2 — The data centre as a heat source

### 2.1 Closing the energy balance

Essentially all electrical power delivered to a data centre leaves as heat. The split that
matters for the receiving water is:

$$\dot Q_{\text{water}} = P_{IT}\cdot \mathrm{PUE}\cdot f_{\text{water}}$$

where $\mathrm{PUE}$ (power usage effectiveness) is total facility power over IT power, and
$f_{\text{water}}$ is the fraction of facility heat rejected to seawater rather than to air.
The cooling-water flow follows from the discharge temperature rise:

$$q = \frac{\dot Q_{\text{water}}}{\rho\,c_p\,\Delta T_{dis}}$$

This is the **design trade-off that drives everything downstream**: for a fixed heat load,
a small $\Delta T_{dis}$ needs a large flow (more entrainment and impingement risk under
CWA §316(b)); a large $\Delta T_{dis}$ needs a small flow but produces a hotter, more
buoyant, harder-to-dilute plume. There is no configuration that avoids both.

### 2.2 Three cooling topologies

| Topology | $\Delta T_{dis}$ | Flow | Regulatory exposure |
|---|---|---|---|
| **Once-through** seawater | 6–11 K | very large | §316(a) thermal + §316(b) intake; CA OTC policy presumption against |
| **Closed loop + seawater heat exchanger** | 8–15 K | moderate | §316(a) thermal; §316(b) if intake > 2 MGD |
| **Sealed capsule, passive conduction** (Natick-class) | n/a — no bulk flow | zero intake | thermal discharge status legally unresolved; heat is still "energy" under UNCLOS Art. 1(1)(4) |

The passive capsule has no intake and therefore no §316(b) exposure, but it also has **no
recoverable heat stream** — which is the finding that decides Part 9.

In [ ]:
PLANT = dict(
    name        = "SET_ME — facility",
    P_IT_MW     = 120.0,     # IT load
    PUE         = 1.15,
    f_to_water  = 0.95,      # fraction of facility heat rejected to seawater
    topology    = "closed_loop_hx",   # once_through | closed_loop_hx | sealed_capsule
    dT_dis      = 8.0,       # K above intake temperature at the port
    n_ports     = 16,        # diffuser ports
    port_D      = 0.40,      # port diameter, m
    x_intake    = 900.0,     # intake distance from the outfall along the tidal axis, m
    z_port      = 3.0,       # height of port above the bed, m
    f_capture   = 0.0,       # fraction of heat captured BEFORE discharge (reuse loop)
)

def dc_heat(P_IT_MW, PUE, f_to_water, dT_dis, T_amb, S_amb, f_capture=0.0):
    P_tot = P_IT_MW*PUE*1e6
    Q_w   = P_tot*f_to_water*(1.0 - f_capture)
    rho   = float(rho_sw(T_amb, S_amb)); cp = float(cp_sw(T_amb, S_amb))
    q     = Q_w/(rho*cp*max(dT_dis, 1e-9))
    return dict(P_total_MW=P_tot/1e6, Q_water_MW=Q_w/1e6, Q_water_W=Q_w,
                q_m3s=q, q_MGD=q*86400*264.172/1e6, rho=rho, cp=cp)

H0 = dc_heat(PLANT["P_IT_MW"], PLANT["PUE"], PLANT["f_to_water"], PLANT["dT_dis"],
             SITE["T_amb_mean"], SITE["S_amb"], PLANT["f_capture"])
print(f"{PLANT['P_IT_MW']:.0f} MW IT x PUE {PLANT['PUE']} -> {H0['P_total_MW']:.1f} MW facility")
print(f"heat to seawater      : {H0['Q_water_MW']:.1f} MW")
print(f"cooling flow at dT={PLANT['dT_dis']:.0f} K : {H0['q_m3s']:.2f} m3/s = {H0['q_MGD']:,.0f} MGD")
_thr = float(CRITERIA.set_index("id").loc["CWA_316b_MGD", "value"])
print(f"CWA 316(b) BTA threshold {_thr} MGD -> "
      f"{'TRIGGERED (best technology available required)' if H0['q_MGD'] > _thr else 'not triggered'}")

# --- the dT_dis / flow trade-off, made explicit ------------------------------
dTs = np.arange(2, 21, 0.5)
rows = [dc_heat(PLANT["P_IT_MW"], PLANT["PUE"], PLANT["f_to_water"], d,
                SITE["T_amb_mean"], SITE["S_amb"]) for d in dTs]
tr = pd.DataFrame(rows); tr["dT_dis"] = dTs

fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
ax[0].plot(dTs, tr.q_MGD, color="#1F3A5F"); ax[0].axhline(_thr, color="crimson", ls="--",
          label=f"316(b) BTA threshold ({_thr:.0f} MGD)")
ax[0].set_yscale("log"); ax[0].set_xlabel("discharge dT (K)"); ax[0].set_ylabel("intake (MGD)")
ax[0].set_title("Small dT -> huge intake -> entrainment risk"); ax[0].legend(fontsize=7)
ax[1].plot(dTs, 9.80665*np.array([float(alpha_sw(SITE['T_amb_mean'], SITE['S_amb']))]*len(dTs))*dTs,
           color="#2E7D82")
ax[1].set_xlabel("discharge dT (K)"); ax[1].set_ylabel("reduced gravity g' (m/s2)")
ax[1].set_title("Large dT -> more buoyant, surfacing plume")
plt.tight_layout(); plt.show()
print("There is no dT that avoids both failure modes. The diffuser design in Part 3 is how")
print("you buy dilution without buying intake volume.")

## Part 3 — Near field: dilution before the plume becomes a passive tracer

### 3.1 The model

Within tens of metres of the port, mixing is driven by the discharge's own momentum and
buoyancy, not by ambient turbulence. Two classical asymptotes bracket it.

**Momentum-dominated (pure jet).** Flux-averaged dilution grows linearly with distance:
$$S_{jet} \approx 0.25\,\frac{z}{D}$$

**Buoyancy-dominated (pure plume).** Starting from $S_p = 0.15\,B^{1/3} z^{5/3}/q$ with
buoyancy flux $B = g'q$, reduced gravity $g' = g\,\alpha\,\Delta T_{dis}$, port area
$A = \pi D^2/4$ and densimetric Froude number $F = U_j/\sqrt{g'D}$, substitution gives

$$S_{plume} = 0.15\left(\tfrac{4}{\pi}\right)^{2/3} F^{-2/3}\left(\tfrac{z}{D}\right)^{5/3}
 \;\approx\; 0.176\,F^{-2/3}\left(\tfrac{z}{D}\right)^{5/3}$$

*(this derivation is reproduced symbolically in the vetting cell so you can check it).*

### 3.2 Why we bound the empirical formulas with a conservation law

Both asymptotes are fits with limited validity — CorJet/CORMIX validation work reports the
jet-integral formulation is unreliable below $F \approx 5$, where the jet falls back on and
re-entrains itself. Rather than extrapolate a fit into a regime where it is known to fail,
we impose a **hard physical ceiling**: you cannot dilute with water that is not there. Over
a mixing footprint of width $b$ and depth $H$ in a current $u_a$, the available ambient flux
is $u_a b H$, so

$$S \le S_{cons} = 1 + \frac{u_a\,b\,H}{q}$$

and we take $S = \min\!\big(\text{blend}(S_{jet},S_{plume}),\,S_{cons}\big)$. This is the
single most important structural decision in the near-field module: **at slack water the
conservation bound binds, and it is the conservation bound — not the empirical fit — that
produces the correct answer that dilution collapses.**

A fit can be wrong. $u_a b H$ cannot.

### Library note — near-field dilution (Correction 1)

`ag.plume.near_field` takes **per-port** fluxes for point plumes and **per-unit-length** fluxes for line plumes (Tian, Roberts & Daviero 2004 regime test on `s/H`). Feeding total diffuser flow into a point-plume law understates dilution by `N^(2/3)`. Coefficients differ from the earlier notebook asymptotes — see [docs/CORRECTIONS.md](../docs/CORRECTIONS.md) item 1 and VERIFICATION P-3.

In [ ]:
# INTEGRATED: aquatic_guardian library
G = 9.80665

def near_field(q, dT_dis, n_ports, D, H, u_amb, b_mix=None, T_amb=14.0, S_amb=31.0,
               z_frac=0.85, jet_c=0.25, plume_c=0.176, port_spacing_m=None):
    """Screening near-field dilution via aquatic_guardian.plume (Correction 1).

    Point-plume laws take PER-PORT fluxes; line-plume laws take flux per unit length.
    Returns a dict compatible with earlier notebook cells, plus library diagnostics.
    """
    # Zero-discharge baseline: no plume to dilute.
    if float(q) <= 0 or int(n_ports) <= 0:
        return dict(U_jet=0.0, F_dens=0.0, g_prime=0.0, alpha=0.0, zD=0.0,
                    S_jet=1.0, S_plume=1.0, S_empirical=1.0, S_conservation=1.0,
                    S=1.0, dT_nf=0.0, b_mix=3.0*H if b_mix is None else b_mix,
                    binding="no_discharge", asymptote_valid=True, regime="point",
                    library=None, total_flow_error_if_misused=1.0)
    # Default spacing: half the depth puts s/H in the transitional band when unknown.
    s = float(port_spacing_m) if port_spacing_m is not None else max(float(D) * 5.0, 0.5 * float(H))
    d = ag.plume.Diffuser(total_flow_m3s=float(q), n_ports=int(n_ports),
                          port_diameter_m=float(D), port_spacing_m=s)
    nf = ag.plume.near_field(d, float(H), float(T_amb), float(dT_dis), float(S_amb),
                             float(u_amb))
    A_port = np.pi * D**2 / 4
    Uj = (q / n_ports) / A_port
    gp = nf["g_prime"]
    Fd = Uj / np.sqrt(max(gp * D, 1e-12))
    S = float(nf["dilution"])
    b_mix = 3.0 * H if b_mix is None else b_mix
    S_cons = 1.0 + (u_amb * b_mix * H) / max(q, 1e-9)
    S_use = max(1.0, min(S, S_cons))
    z = z_frac * H
    return dict(U_jet=Uj, F_dens=Fd, g_prime=gp, alpha=gp / (G * max(dT_dis, 1e-12)),
                zD=z / D, S_jet=nf.get("dilution_point", S), S_plume=nf.get("dilution_line", S),
                S_empirical=S, S_conservation=S_cons, S=S_use, dT_nf=dT_dis / S_use,
                b_mix=b_mix, binding="conservation" if S_cons < S else "library",
                asymptote_valid=True, regime=nf["regime"], library=nf,
                total_flow_error_if_misused=ag.plume.total_flow_error_factor(int(n_ports)))

# ---- VET 1: per-port flux — Correction 1 factor is exactly N^(2/3) ----------
_n = 16
_factor = ag.plume.total_flow_error_factor(_n)
assert abs(_factor - _n**(2/3)) < 1e-12
print(f"[VET] substituting total flow into a point-plume law understates dilution by "
      f"N^(2/3) = {_factor:.2f} for N={_n} ports (Correction 1) -> PASS")

# ---- VET 2: library stagnant dilution is finite and >= 1 -------------------
_d = ag.plume.Diffuser(4.0, 16, 0.4, 2.0)
_nf = ag.plume.near_field(_d, 14.0, 11.5, 8.0, 30.5, 0.0)
assert _nf["dilution"] >= 1.0
print(f"[VET] library near_field dilution={_nf['dilution']:.2f}, regime={_nf['regime']} "
      f"-> PASS")
print("Legacy jet/plume regrouping constants (0.176 / 0.25) are NOT used; the library "
      "uses Fischer et al. (1979) C_point=0.089 and C_line=0.38 (needs UM3 benchmark, P-3).")


### 3.3 Diffuser design as a compliance instrument

Dilution scales roughly as $z/D$, so splitting the same flow across more, smaller ports buys
dilution with no extra intake volume. This is the cheapest available mitigation and the first
thing a reviewer will ask about. The sweep below answers "how many ports do I need to hold
the near-field boundary below the CCME ±1 °C criterion at the 10th-percentile current?"

In [ ]:
u_lo = float(np.percentile(np.abs(tidal_current(np.arange(0, 15*86400, 300.),
                                                SITE["U_amps"], SITE["U_phases"])), 10))
rows = []
for n in [1, 2, 4, 8, 16, 24, 32, 48, 64]:
    D = np.sqrt(4*H0["q_m3s"]/(n*np.pi*2.6))          # size ports for ~2.6 m/s exit velocity
    for lab, uu in [("p10 current", u_lo), ("median", U_eff*0.64), ("peak", U_eff)]:
        r = near_field(H0["q_m3s"], PLANT["dT_dis"], n, D, SITE["depth_H"], uu,
                       T_amb=SITE["T_amb_mean"], S_amb=SITE["S_amb"])
        rows.append(dict(n_ports=n, port_D_m=round(D, 3), current=lab, u=round(uu, 3),
                         F=round(r["F_dens"], 1), S=round(r["S"], 1),
                         dT_nf_K=round(r["dT_nf"], 2), bound=r["binding"],
                         valid=r["asymptote_valid"]))
dif = pd.DataFrame(rows)
piv = dif.pivot(index="n_ports", columns="current", values="dT_nf_K")[["p10 current", "median", "peak"]]
display(piv.style.background_gradient(cmap="RdYlGn_r", axis=None).format("{:.2f}")
        .set_caption("dT (K) at the near-field boundary — CCME limit is 1.0 K"))

ok = piv[piv["p10 current"] <= 1.0]
print(f"\nSmallest diffuser meeting CCME 1 K at the 10th-percentile current: "
      f"{ok.index.min() if len(ok) else 'NONE of the tested configurations'} ports")
print("Rows flagged valid=False sit outside the jet-integral validity window (F<5 or z/D<8);")
print("there the conservation bound is doing the work, which is the intended behaviour.")
display(dif[~dif["valid"]].head(8))

## Part 4 — Far field: a solver you can trust

### 4.1 Governing equation

Beyond the near field the excess temperature $\theta = T - T_{amb}$ is a passive scalar:

$$\frac{\partial\theta}{\partial t} + u(t)\frac{\partial\theta}{\partial x}
 + v(t)\frac{\partial\theta}{\partial y}
 = K_x\frac{\partial^2\theta}{\partial x^2} + K_y\frac{\partial^2\theta}{\partial y^2}
 - \lambda\theta + S(x,y)$$

with $\lambda = K_{surf}/(\rho c_p H)$ the surface-loss rate and $u(t)$ the harmonic tidal
current from Part 1.

### 4.2 Why the scheme was rebuilt

A first-order upwind + FTCS scheme — the obvious choice, and the one in the original
scaffold — introduces **numerical diffusion** with an effective coefficient

$$K_{num} \approx \frac{\langle|u|\rangle\,\Delta x}{2}\left(1 - C\right),\qquad C=\frac{u\Delta t}{\Delta x}$$

At $\Delta x = 25$ m and $\langle|u|\rangle = 2U_0/\pi \approx 0.19$ m/s this is
$\approx 2.2$ m²/s — roughly **20 % of the physical dispersion we are trying to model**,
and it is *invisible* unless you test for it. A plume model that silently adds a fifth of its
own mixing will under-predict both peak temperature and compliance-relevant gradients.

The replacement uses **Strang operator splitting**:

1. half-step source + decay — solved *exactly* by the exponential integrator
   $\theta \leftarrow \theta e^{-\lambda \Delta t/2} + \frac{S}{\lambda}(1-e^{-\lambda\Delta t/2})$
2. half-step advection — **semi-Lagrangian**. Because $u(t)$ is uniform in space, the
   advection step is an exact translation; only the cubic-Lagrange interpolation introduces
   error, and it introduces no first-order diffusion
3. full-step diffusion — **Crank–Nicolson**, unconditionally stable, solved with a banded
   tridiagonal factorisation
4. half-step advection, 5. half-step source + decay

giving second-order accuracy in space and time with no CFL restriction on stability.

### Library note — numerical diffusion (Correction 3)

First-order upwind adds `K_num = |u|Δx/2 (1−C)`, often larger than Okubo physical diffusivity below ~3 km. Use a flux-limited scheme (`ag.numerics.advect_diffuse_1d(..., scheme='tvd')`) or report `K_num` beside `ag.numerics.okubo_kh(L)`. See [docs/CORRECTIONS.md](../docs/CORRECTIONS.md) item 3.

In [ ]:
# INTEGRATED: aquatic_guardian library
def advect_semilag_1d(th, dx, shift):
    '''Exact uniform translation with cubic-Lagrange interpolation (no first-order diffusion).'''
    n = th.size
    s = np.arange(n) - shift/dx
    i0 = np.floor(s).astype(int); f = s - i0
    def g(idx):
        ok = (idx >= 0) & (idx < n)
        return np.where(ok, th[np.clip(idx, 0, n-1)], 0.0)
    out = (-f*(f-1)*(f-2)/6.0)*g(i0-1) + ((f+1)*(f-1)*(f-2)/2.0)*g(i0) \
          + (-(f+1)*f*(f-2)/2.0)*g(i0+1) + ((f+1)*f*(f-1)/6.0)*g(i0+2)
    return np.where((s >= 0) & (s <= n-1), out, 0.0)

def diffuse_cn_1d(th, dx, dt, K, neumann=False):
    '''Crank-Nicolson diffusion. neumann=True conserves the integral exactly.'''
    n = th.size; r = K*dt/(2*dx*dx)
    ab = np.zeros((3, n)); ab[0, 1:] = -r; ab[1, :] = 1+2*r; ab[2, :-1] = -r
    rhs = th.copy(); rhs[1:-1] = r*th[:-2] + (1-2*r)*th[1:-1] + r*th[2:]
    if neumann:
        ab[1, 0] = 1+r; ab[1, -1] = 1+r
        rhs[0] = (1-r)*th[0] + r*th[1]; rhs[-1] = (1-r)*th[-1] + r*th[-2]
    else:
        ab[1, 0] = 1.0; ab[0, 1] = 0.0; ab[1, -1] = 1.0; ab[2, -2] = 0.0
        rhs[0] = 0.0; rhs[-1] = 0.0
    return solve_banded((1, 1), ab, rhs)

def step_source_decay(th, S, lam, dt):
    if lam > 1e-20:
        return th*np.exp(-lam*dt) + (S/lam)*(1 - np.exp(-lam*dt))
    return th + S*dt

def solve_1d(L_half, dx, dt, n_cycles, U_amps, U_phases, K, lam, S_field,
             recorders=None, snapshot_phases=(0.0, .25, .5, .75)):
    '''Strang-split 1-D advection-diffusion-decay over n_cycles M2 periods.'''
    x = np.arange(-L_half, L_half+dx, dx); n = x.size
    S = S_field(x)
    nst = int(round(n_cycles*T_M2/dt))
    th = np.zeros(n)
    ridx = {k: int(np.argmin(np.abs(x-v))) for k, v in (recorders or {}).items()}
    ser = {k: np.zeros(nst) for k in ridx}
    snaps, snap_at = {}, {p: int(((n_cycles-1+p)*T_M2)/dt) for p in snapshot_phases}
    th_max = np.zeros(n)
    for k in range(nst):
        t = k*dt
        th = step_source_decay(th, S, lam, dt/2)
        th = advect_semilag_1d(th, dx, float(tidal_current(t+dt*.25, U_amps, U_phases))*dt/2)
        th = diffuse_cn_1d(th, dx, dt, K)
        th = advect_semilag_1d(th, dx, float(tidal_current(t+dt*.75, U_amps, U_phases))*dt/2)
        th = step_source_decay(th, S, lam, dt/2)
        th[0] = th[-1] = 0.0
        if t >= (n_cycles-1)*T_M2:
            th_max = np.maximum(th_max, th)
        for kk, i in ridx.items():
            ser[kk][k] = th[i]
        for p, ss in snap_at.items():
            if k == ss: snaps[p] = th.copy()
    return dict(x=x, theta=th, theta_max=th_max, t=np.arange(nst)*dt,
                series=ser, snaps=snaps, dt=dt, nsteps=nst)

print("Solver defined. Verification suite runs next — it must pass before any result is used.")

# Library reference: flux-limited TVD + numerical-diffusion diagnostics (Correction 3)
_dx, _u, _dt = 25.0, 0.30, 30.0
_K_num = ag.numerics.k_num_upwind(_u, _dx, _dt)
_K_okubo_1km = ag.numerics.okubo_kh(1000.0)
print(f"[library] K_num upwind at dx={_dx} m, u={_u} m/s: {_K_num:.2f} m2/s "
      f"(Okubo at 1 km: {_K_okubo_1km:.2f} m2/s; crossover ~"
      f"{ag.numerics.okubo_crossover_scale(_K_num):.0f} m)")
_x = np.linspace(-500, 500, 41)
_c0 = np.exp(-(_x / 40.0)**2)
_c_tvd = ag.numerics.advect_diffuse_1d(_c0, _u, 0.0, _dx, _dt, n_steps=20, scheme="tvd")
_c_up = ag.numerics.advect_diffuse_1d(_c0, _u, 0.0, _dx, _dt, n_steps=20, scheme="upwind")
assert _c_tvd.min() >= -1e-12 and abs(_c_tvd.sum() - _c0.sum()) < 1e-6
print(f"[library] TVD peak={{_c_tvd.max():.3f}} vs upwind peak={{_c_up.max():.3f}} "
      "(TVD retains more peak — Correction 3) -> PASS")
print("This notebook's production far-field solver remains cubic semi-Lagrange + CN "
      "(already free of first-order upwind diffusion); the library call documents parity "
      "with the screening package.")


## Part 5 — Verification suite

> **This is the part that makes the rest of the notebook usable.** A model that has not been
> tested against something it cannot fake is a drawing, not a measurement. Each test below
> has an `assert`: if you change the solver and break it, the notebook stops.

| # | Test | What it catches |
|---|---|---|
| V1 | Advected–diffusing Gaussian vs closed-form solution | wrong advection, wrong diffusion, wrong splitting order |
| V2 | Grid convergence order | scheme silently degraded to first order |
| V3 | Integral conservation with no-flux ends | mass/energy leaking |
| V4 | Steady point source in a uniform current vs closed form | wrong decay coupling, wrong boundary treatment |
| V5 | Numerical-diffusion audit of the old upwind scheme | quantifies what we fixed |

### V1 — closed-form benchmark

For $u(t)=U_0\sin(2\pi t/T)$ and no source, a Gaussian initial condition stays Gaussian:

$$\theta(x,t)=\frac{M}{\sqrt{2\pi(\sigma_0^2+2Kt)}}
 \exp\!\left[-\frac{(x-X(t))^2}{2(\sigma_0^2+2Kt)}\right]e^{-\lambda t},
 \qquad X(t)=\frac{U_0T}{2\pi}\left(1-\cos\frac{2\pi t}{T}\right)$$

The centre follows the tidal displacement, the variance grows as $2Kt$, the amplitude decays
as $e^{-\lambda t}$. All three must be reproduced simultaneously.

In [ ]:
K_V, LAM_V, U0_V, S0_V = 12.0, 7.3e-7, 0.30, 150.0

def exact_gauss(x, t):
    X  = (U0_V*T_M2/(2*np.pi))*(1 - np.cos(2*np.pi*t/T_M2))
    s2 = S0_V**2 + 2*K_V*t
    return np.exp(-(x-X)**2/(2*s2))/np.sqrt(2*np.pi*s2)*np.exp(-LAM_V*t)

def run_bench(dx, dt, t_end, neumann=False):
    L = 9000.0; x = np.arange(-L, L+dx, dx); th = exact_gauss(x, 0.0); t = 0.0
    while t < t_end - 1e-9:
        s = min(dt, t_end-t)
        th = th*np.exp(-LAM_V*s/2)
        th = advect_semilag_1d(th, dx, U0_V*np.sin(2*np.pi*(t+s*.25)/T_M2)*s/2)
        th = diffuse_cn_1d(th, dx, s, K_V, neumann=neumann)
        th = advect_semilag_1d(th, dx, U0_V*np.sin(2*np.pi*(t+s*.75)/T_M2)*s/2)
        th = th*np.exp(-LAM_V*s/2)
        t += s
    return x, th

t_end = 0.75*T_M2
rows, prev = [], None
for dx, dt in [(100., 600.), (50., 300.), (25., 150.), (12.5, 75.), (6.25, 37.5)]:
    x, th = run_bench(dx, dt, t_end); ex = exact_gauss(x, t_end)
    L2 = np.sqrt(np.sum((th-ex)**2)*dx)/np.sqrt(np.sum(ex**2)*dx)
    rows.append(dict(dx_m=dx, dt_s=dt, rel_L2=L2,
                     observed_order=np.nan if prev is None else np.log2(prev/L2)))
    prev = L2
conv = pd.DataFrame(rows)
display(conv.style.format({"rel_L2": "{:.3e}", "observed_order": "{:.2f}"}))
assert conv.rel_L2.iloc[2] < 2e-4, "V1 FAILED: solver does not match the analytic solution"
assert conv.observed_order.iloc[-1] > 1.7, "V2 FAILED: convergence order below second"
print(f"[VET V1] relative L2 error at dx=25 m: {conv.rel_L2.iloc[2]:.2e}  -> PASS")
print(f"[VET V2] observed convergence order -> {conv.observed_order.iloc[-1]:.2f} (target 2) -> PASS")

fig, ax = plt.subplots(1, 2, figsize=(11, 3.3))
x, th = run_bench(25., 150., t_end)
ax[0].plot(x/1000, exact_gauss(x, t_end), lw=3, alpha=.35, color="k", label="analytic")
ax[0].plot(x/1000, th, lw=1.1, color="#C0392B", label="solver (dx=25 m)")
ax[0].set_xlim(-2, 6); ax[0].set_xlabel("x (km)"); ax[0].set_ylabel("theta")
ax[0].legend(fontsize=8); ax[0].set_title("V1 — advected, diffusing, decaying Gaussian")
ax[1].loglog(conv.dx_m, conv.rel_L2, "o-", color="#1F3A5F", label="observed")
ax[1].loglog(conv.dx_m, conv.rel_L2.iloc[-1]*(conv.dx_m/conv.dx_m.iloc[-1])**2,
             "--", color="#888", label="2nd-order reference")
ax[1].set_xlabel("dx (m)"); ax[1].set_ylabel("relative L2"); ax[1].legend(fontsize=8)
ax[1].set_title("V2 — grid convergence")
plt.tight_layout(); plt.show()

In [ ]:
# ---- V3: integral conservation with no-flux boundaries ----------------------
dx, dt = 25.0, 150.0
L = 15000.0; x = np.arange(-L, L+dx, dx)
th = np.exp(-x**2/(2*200.0**2))/np.sqrt(2*np.pi*200.0**2)
m0 = np.trapezoid(th, x); t = 0.0
while t < 2*T_M2 - 1e-9:
    s = min(dt, 2*T_M2-t)
    th = advect_semilag_1d(th, dx, U0_V*np.sin(2*np.pi*(t+s*.25)/T_M2)*s/2)
    th = diffuse_cn_1d(th, dx, s, K_V, neumann=True)
    th = advect_semilag_1d(th, dx, U0_V*np.sin(2*np.pi*(t+s*.75)/T_M2)*s/2)
    t += s
m1 = np.trapezoid(th, x)
assert abs(m1/m0 - 1) < 1e-6, "V3 FAILED: the solver is not conserving"
print(f"[VET V3] integral drift over two tidal cycles: {100*(m1/m0-1):+.2e} %  -> PASS")

# ---- V4: steady point source in a uniform current --------------------------
u_s, K_s, lam_s, q_s = 0.12, 12.0, 7.3e-7, 1.0
disc = np.sqrt(u_s**2 + 4*K_s*lam_s); Aan = q_s/disc
rp, rm = (u_s-disc)/(2*K_s), (u_s+disc)/(2*K_s)
dx = 10.0; L = 6000.0; xs = np.arange(-L, L+dx, dx)
out = []
for sig in [80., 40., 20.]:
    S = q_s*np.exp(-xs**2/(2*sig**2))/(sig*np.sqrt(2*np.pi))
    th = np.zeros_like(xs); dt = 40.0
    for k in range(int(9e5/dt)):
        th = step_source_decay(th, S, lam_s, dt/2)
        th = advect_semilag_1d(th, dx, u_s*dt/2)
        th = diffuse_cn_1d(th, dx, dt, K_s)
        th = advect_semilag_1d(th, dx, u_s*dt/2)
        th = step_source_decay(th, S, lam_s, dt/2)
        th[0] = th[-1] = 0.0
    for xx in [-400, -100, 500, 3000]:
        an = Aan*np.exp(rp*xx) if xx >= 0 else Aan*np.exp(rm*xx)
        out.append(dict(source_sigma_m=sig, x_m=xx,
                        numeric=th[int(np.argmin(np.abs(xs-xx)))], analytic=an))
v4 = pd.DataFrame(out); v4["rel_err_%"] = 100*(v4.numeric/v4.analytic - 1)
display(v4.pivot(index="x_m", columns="source_sigma_m", values="rel_err_%").round(2)
        .style.set_caption("V4 — % error vs the delta-source analytic solution"))
ds = v4[v4.x_m > 0]
assert ds["rel_err_%"].abs().max() < 0.5, "V4 FAILED downstream"
print(f"[VET V4] downstream error < {ds['rel_err_%'].abs().max():.2f} %  -> PASS")
print("Upstream error shrinks as the numerical source narrows toward a true delta function,")
print("which is the expected behaviour and confirms the boundary treatment is not the cause.")

In [ ]:
# INTEGRATED: aquatic_guardian library
# ---- V5: numerical-diffusion audit of the scheme we replaced ----------------
def run_upwind(dx, t_end, cfl=0.35):
    L = 9000.0; x = np.arange(-L, L+dx, dx); N = x.size
    th = exact_gauss(x, 0.0)
    dt = cfl*min(dx/U0_V, dx*dx/(2*K_V), 1/LAM_V)
    nst = int(t_end/dt)
    for n in range(nst):
        u = U0_V*np.sin(2*np.pi*(n*dt)/T_M2)
        adv = np.zeros(N)
        if u >= 0: adv[1:]  = -u*(th[1:]-th[:-1])/dx
        else:      adv[:-1] = -u*(th[1:]-th[:-1])/dx
        dif = np.zeros(N); dif[1:-1] = K_V*(th[2:]-2*th[1:-1]+th[:-2])/dx**2
        th = th + dt*(adv+dif-LAM_V*th); th[0] = th[-1] = 0.0
    return x, th, dt*nst

def variance_of(x, th):
    m0 = np.trapezoid(th, x); m1 = np.trapezoid(x*th, x)/m0
    return np.trapezoid((x-m1)**2*th, x)/m0

rows = []
for dx in [50., 25., 12.5]:
    x, thu, tr = run_upwind(dx, t_end)
    ex = exact_gauss(x, tr)
    K_num = (variance_of(x, thu) - variance_of(x, ex))/(2*tr)
    u_mean = 2*U0_V/np.pi
    rows.append(dict(dx_m=dx, K_num_measured=K_num,
                     K_num_theory=u_mean*dx/2, pct_of_physical_K=100*K_num/K_V))
v5 = pd.DataFrame(rows)
display(v5.round(3).style.set_caption(
    "V5 — spurious diffusion of first-order upwind, measured vs the modified-equation prediction"))
print(f"[VET V5] at dx=25 m the old scheme added {v5.pct_of_physical_K.iloc[1]:.0f}% of the")
print("physical dispersion as numerical artefact. Measured tracks the theoretical")
print("<|u|> dx / 2 (with the (1-C) correction), confirming the diagnosis.")
print("\nConsequence for decisions: a model carrying ~20% extra mixing under-predicts peak")
print("near-outfall dT and flattens the gradients that compliance criteria are written on.")

# Library closed-form beside the measured audit
for _, row in v5.iterrows():
    _kn = ag.numerics.k_num_upwind(2*U0_V/np.pi, row.dx_m, None)  # mean |u| form without C
    print(f"[library] dx={{row.dx_m}}: K_num formula (C=0)={{_kn:.2f}} m2/s; "
          f"measured={{row.K_num_measured:.2f}}; Okubo@1km={{ag.numerics.okubo_kh(1000):.2f}}")


## Part 6 — Scenarios and the 2-D plume footprint

### 6.1 From near field to far field

The near-field module hands the far field a **residual heat load** and a source footprint.
The far-field source strength in K/s is

$$S(x,y) = \frac{\dot Q_{water}(1-f_{capture})}{\rho c_p H}\, g_\sigma(x,y),
\qquad \iint g_\sigma \,dA = 1$$

with $\sigma_s$ set by the near-field footprint (order $3H$ for a diffuser).

A 2-D depth-averaged solve is worth the cost because **regulations are written on areas and
distances**, not on point values: "ΔT ≤ 4 °F beyond 1 000 ft", "area of the mixing zone",
"7-DADM at the edge of the zone of initial dilution". A 1-D model cannot produce an area.

In [ ]:
def cn_axis_2d(F, dx, dt, K, axis):
    '''Crank-Nicolson diffusion along one axis of a 2-D field, vectorised over the other.'''
    F = np.moveaxis(F, axis, 0); n = F.shape[0]
    r = K*dt/(2*dx*dx)
    ab = np.zeros((3, n)); ab[0, 1:] = -r; ab[1, :] = 1+2*r; ab[2, :-1] = -r
    ab[1, 0] = 1+r; ab[1, -1] = 1+r
    rhs = F.copy()
    rhs[1:-1] = r*F[:-2] + (1-2*r)*F[1:-1] + r*F[2:]
    rhs[0] = (1-r)*F[0] + r*F[1]; rhs[-1] = (1-r)*F[-1] + r*F[-2]
    return np.moveaxis(solve_banded((1, 1), ab, rhs), 0, axis)

def advect_2d(F, dx, dy, ux, uy, dt):
    ny, nx = F.shape
    j = np.arange(nx)[None, :] - ux*dt/dx
    i = np.arange(ny)[:, None] - uy*dt/dy
    j0 = np.floor(j).astype(int); i0 = np.floor(i).astype(int)
    fx, fy = j-j0, i-i0
    def g(ii, jj):
        ok = (ii >= 0) & (ii < ny) & (jj >= 0) & (jj < nx)
        return np.where(ok, F[np.clip(ii, 0, ny-1), np.clip(jj, 0, nx-1)], 0.0)
    i0b = np.broadcast_to(i0, F.shape); j0b = np.broadcast_to(j0, F.shape)
    fxb = np.broadcast_to(fx, F.shape); fyb = np.broadcast_to(fy, F.shape)
    return ((1-fyb)*((1-fxb)*g(i0b, j0b) + fxb*g(i0b, j0b+1))
            + fyb*((1-fxb)*g(i0b+1, j0b) + fxb*g(i0b+1, j0b+1)))

def solve_2d(Q_W, H, dx=40., dy=40., nx=201, ny=101, K_x=12., K_y=3., lam=7.3e-7,
             sigma_s=45., U_amps=None, U_phases=None, axis_deg=0., n_cycles=3, dt=180.,
             T_amb=11.5, S_amb=30.5, track_thresholds=(0.3, 1.0, 2.2),
             u_steady=None, t_end=None):
    '''If u_steady is given the current is held constant (regulatory mixing-zone mode);
    otherwise the full harmonic tide is used. t_end overrides n_cycles*T_M2.'''
    x = (np.arange(nx)-nx//2)*dx; y = (np.arange(ny)-ny//2)*dy
    Xg, Yg = np.meshgrid(x, y)
    g = np.exp(-(Xg**2+Yg**2)/(2*sigma_s**2)); g /= g.sum()*dx*dy
    rho, cp = float(rho_sw(T_amb, S_amb)), float(cp_sw(T_amb, S_amb))
    S = Q_W/(rho*cp*H)*g
    th = np.zeros((ny, nx)); th_max = np.zeros_like(th)
    ang = np.deg2rad(axis_deg)
    T_run = t_end if t_end is not None else n_cycles*T_M2
    nst = int(round(T_run/dt))
    areas = {k: 0.0 for k in track_thresholds}
    for k in range(nst):
        t = k*dt
        u = float(u_steady) if u_steady is not None else float(tidal_current(t, U_amps, U_phases))
        ux, uy = u*np.cos(ang), u*np.sin(ang)
        th = step_source_decay(th, S, lam, dt/2)
        th = advect_2d(th, dx, dy, ux, uy, dt/2)
        th = cn_axis_2d(th, dx, dt, K_x, 1)
        th = cn_axis_2d(th, dy, dt, K_y, 0)
        th = advect_2d(th, dx, dy, ux, uy, dt/2)
        th = step_source_decay(th, S, lam, dt/2)
        th[0, :] = th[-1, :] = th[:, 0] = th[:, -1] = 0.0
        if t >= 0.5*T_run:
            th_max = np.maximum(th_max, th)
            for thr in track_thresholds:
                areas[thr] = max(areas[thr], (th > thr).sum()*dx*dy)
    return dict(x=x, y=y, theta=th, theta_max=th_max, areas_m2=areas, dx=dx, dy=dy)

# ---- VET A: does the SCHEME conserve? (no advection, so no domain-size confound) ----
_rows = []
for _dx, _dt, _sig in [(60., 300., 100.), (40., 200., 100.), (25., 150., 100.), (15., 100., 100.)]:
    _n = int(2500/_dx)*2 + 1
    _r = solve_2d(160e6, 14.0, dx=_dx, dy=_dx, nx=_n, ny=_n, lam=0.0, sigma_s=_sig,
                  u_steady=0.0, t_end=6*3600., dt=_dt)
    _E = _r["theta"].sum()*_dx*_dx*float(rho_sw(11.5, 30.5))*float(cp_sw(11.5, 30.5))*14.0
    _rows.append(dict(dx_m=_dx, cells_per_sigma=round(_sig/_dx, 2),
                      grid=f"{_n}x{_n}", energy_retained=_E/(160e6*6*3600.)))
_v2d = pd.DataFrame(_rows)
display(_v2d.round(4).style.set_caption(
    "VET A — source + Crank-Nicolson diffusion only, lam=0: ALL injected heat must stay"))
assert _v2d.energy_retained.min() > 0.99, "the 2-D scheme itself is losing energy"
print(f"[VET 2-D A] the scheme conserves to better than "
      f"{100*(1-_v2d.energy_retained.min()):.2f}% at every resolution -> PASS")

# ---- VET B: the constraint that actually bites is DOMAIN SIZE -----------------
_t = np.arange(0, 2*T_M2, 60.)
_disp = np.cumsum(tidal_current(_t, SITE["U_amps"], SITE["U_phases"]))*60.
_maxdisp = float(np.abs(_disp).max())
print(f"\n[VET 2-D B] a water parcel wanders {_maxdisp:,.0f} m over two tidal cycles.")
_rows = []
for _half in [3000., 6000., 12000.]:
    _dx = 50.; _n = int(_half/_dx)*2 + 1
    _r = solve_2d(160e6, 14.0, dx=_dx, dy=_dx, nx=_n, ny=max(61, _n//3), lam=0.0, sigma_s=100.,
                  U_amps=SITE["U_amps"], U_phases=SITE["U_phases"], n_cycles=0.5, dt=200.)
    _E = _r["theta"].sum()*_dx*_dx*float(rho_sw(11.5, 30.5))*float(cp_sw(11.5, 30.5))*14.0
    _rows.append(dict(half_width_km=_half/1000, energy_retained=_E/(160e6*0.5*T_M2)))
display(pd.DataFrame(_rows).round(4).style.set_caption(
    "VET B — with the full tide, heat leaves a domain narrower than the tidal excursion"))
print("A tidally-forced 2-D domain must be wider than the excursion, which for this site means")
print(f"a half-width above ~{_maxdisp/1000:.0f} km. At 25 m resolution that is a ~1000x500 grid and")
print("minutes per scenario on a free runtime.")
print("\nSo the two solvers are given different jobs, which is also what practice does:")
print("  1-D  -> the full tidal dynamics: recirculation, rate-of-change, time-above-threshold")
print("  2-D  -> the regulatory MIXING-ZONE FOOTPRINT under a steady design current, which is")
print("          how CORMIX-class analyses and 316(a) demonstrations are actually framed.")
print("The 2-D runs below therefore use the 10th-percentile and median currents as steady")
print("design conditions, stated explicitly rather than hidden inside a tidal average.")

### 6.2 The scenario set

Three **core** scenarios answer the charter's question directly, and five **adverse** cases
probe the conditions that turn a mild plume into a problem. The adverse cases are not
pessimism for its own sake — each corresponds to a documented failure mode from the
project's risk register.

In [ ]:
def scenario_config(**over):
    base = dict(P_IT_MW=PLANT["P_IT_MW"], PUE=PLANT["PUE"], f_to_water=PLANT["f_to_water"],
                dT_dis=PLANT["dT_dis"], n_ports=PLANT["n_ports"], port_D=PLANT["port_D"],
                H=SITE["depth_H"], K_x=SITE["K_long"], K_y=SITE["K_lat"],
                K_surf=32.0, U_scale=1.0, f_capture=0.0,
                T_amb=SITE["T_amb_mean"], S_amb=SITE["S_amb"], x_intake=PLANT["x_intake"])
    base.update(over); return base

def run_scenario(cfg, do_2d=True, n_cycles=3):
    hb = dc_heat(cfg["P_IT_MW"], cfg["PUE"], cfg["f_to_water"], cfg["dT_dis"],
                 cfg["T_amb"], cfg["S_amb"], cfg["f_capture"])
    amps = {k: v*cfg["U_scale"] for k, v in SITE["U_amps"].items()}
    u_p10 = float(np.percentile(np.abs(tidal_current(np.arange(0, 15*86400, 300.),
                                                     amps, SITE["U_phases"])), 10))
    nf = near_field(hb["q_m3s"], cfg["dT_dis"], cfg["n_ports"], cfg["port_D"], cfg["H"],
                    max(u_p10, 0.01), T_amb=cfg["T_amb"], S_amb=cfg["S_amb"])
    lam = cfg["K_surf"]/(hb["rho"]*hb["cp"]*cfg["H"])
    sigma_s = max(3.0*cfg["H"], 100.0)   # near-field footprint: diffuser span + spreading
    S_amp = hb["Q_water_W"]/(hb["rho"]*hb["cp"]*cfg["H"]*np.sqrt(2*np.pi)*sigma_s)
    dx = 40. if RUN["FAST"] else 25.
    dt = 180. if RUN["FAST"] else 120.
    r1 = solve_1d(L_half=12000., dx=dx, dt=dt, n_cycles=n_cycles, U_amps=amps,
                  U_phases=SITE["U_phases"], K=cfg["K_x"], lam=lam,
                  S_field=lambda x: S_amp*np.exp(-x**2/(2*sigma_s**2)),
                  recorders={"outfall": 0.0, "intake": cfg["x_intake"],
                             "m250": 250., "m1000": 1000.})
    mask = r1["t"] >= (n_cycles-1)*T_M2
    out_m = r1["series"]["outfall"][mask].mean()
    recirc = float(r1["series"]["intake"][mask].mean()/max(out_m, 1e-12))
    dth_dt = np.abs(np.gradient(r1["series"]["m250"][mask], r1["dt"]))*3600.
    res = dict(
        Q_MW=hb["Q_water_MW"], q_m3s=hb["q_m3s"], q_MGD=hb["q_MGD"],
        S_nf=nf["S"], dT_nf_K=nf["dT_nf"], nf_bound=nf["binding"],
        dT_far_max_K=float(r1["theta_max"].max()),
        dT_500m_K=float(r1["theta_max"][int(np.argmin(np.abs(r1["x"]-500)))]),
        dT_1km_K=float(r1["theta_max"][int(np.argmin(np.abs(r1["x"]-1000)))]),
        recirc_index=recirc, dT_intake_K=float(r1["series"]["intake"][mask].max()),
        rate_max_K_per_h=float(dth_dt.max()),
        frac_time_above_0p3=float((r1["series"]["m250"][mask] > 0.3).mean()),
        lam=lam, sigma_s=sigma_s, u_p10=u_p10)
    if do_2d:
        # regulatory mixing-zone footprint under a STEADY design current (see VET B).
        dx2 = max(12.0, sigma_s/3.0)          # >= 3 cells per source sigma
        u_design = max(u_p10, 0.02)
        half2 = max(2500.0, 1.4*u_design*8*3600.)
        n2 = int(half2/dx2)*2 + 1
        r2 = solve_2d(hb["Q_water_W"], cfg["H"], dx=dx2, dy=dx2, nx=n2, ny=max(81, n2//3),
                      K_x=cfg["K_x"], K_y=cfg["K_y"], lam=lam, sigma_s=sigma_s,
                      u_steady=u_design, t_end=8*3600., dt=180.,
                      T_amb=cfg["T_amb"], S_amb=cfg["S_amb"])
        res["u_design_ms"] = u_design
        res["area_gt0p3_ha"] = r2["areas_m2"][0.3]/1e4
        res["area_gt1p0_ha"] = r2["areas_m2"][1.0]/1e4
        res["_2d"] = r2
    res["_1d"] = r1
    return res

SCENARIOS = {
  "S0_baseline_no_discharge": dict(P_IT_MW=0.0),
  "S1_discharge_only":        dict(),
  "S2_reuse_capture_60":      dict(f_capture=0.60),
  "A1_slack_low_exchange":    dict(U_scale=0.35, K_x=4.0, K_y=1.0),
  "A2_marine_heatwave":       dict(K_surf=15.0, T_amb=SITE["T_amb_summer_7DADM"]),
  "A3_chiller_fault":         dict(dT_dis=13.0),
  "A4_diffuser_fouled":       dict(n_ports=4, port_D=0.80),
  "A5_intake_too_close":      dict(x_intake=250.0),
}

t0 = time.time(); RES = {}
for nm, ov in SCENARIOS.items():
    RES[nm] = run_scenario(scenario_config(**ov), do_2d=True)
print(f"8 scenarios (1-D + 2-D) in {time.time()-t0:.1f} s")

cols = ["Q_MW", "q_MGD", "S_nf", "dT_nf_K", "nf_bound", "dT_far_max_K", "dT_500m_K",
        "dT_1km_K", "dT_intake_K", "recirc_index", "rate_max_K_per_h",
        "u_design_ms", "area_gt0p3_ha", "area_gt1p0_ha"]
SUM = pd.DataFrame({k: {c: v.get(c) for c in cols} for k, v in RES.items()}).T
num = SUM.drop(columns=["nf_bound"]).astype(float).round(3)
num["nf_bound"] = SUM["nf_bound"]
display(num)

### 6.3 Reading the table honestly

Three patterns should be visible, and each is a real result rather than a modelling artefact:

1. **Far-field ΔT is small; near-field ΔT is not.** Tidal flushing disperses a 130 MW load
   into hundredths of a kelvin a kilometre away. This is consistent with the Natick Phase 2
   finding of thousandths of a degree a few metres downstream and HiCloud's reported < 1 °C
   around a test pod. The ecological question is therefore **not** "does the bay warm up" —
   it is "what happens inside the mixing zone, and what happens at slack water".
2. **The binding constraint migrates.** At peak flow the empirical jet/plume asymptote sets
   dilution; at slack the mass-conservation ceiling does. Compliance is decided in the
   hours the facility's own brochure will not show you.
3. **Recirculation is a geometry problem, not a thermal one.** It depends on where the
   intake is relative to $L_{ex}$, and it feeds back into the plant: re-entrained warm water
   raises intake temperature, which raises discharge temperature, which raises the plume.

In [ ]:
fig = plt.figure(figsize=(13, 7.2))
gs = fig.add_gridspec(2, 3, hspace=.42, wspace=.28)

ax = fig.add_subplot(gs[0, :2])
r = RES["S1_discharge_only"]["_1d"]
for p, c in zip([0.0, .25, .5, .75], ["#1F3A5F", "#2E7D82", "#C0392B", "#8E6C1F"]):
    ax.plot(r["x"]/1000, r["snaps"][p], lw=1.2, color=c, label=f"phase {p:.2f} T")
ax.plot(r["x"]/1000, r["theta_max"], "k--", lw=1, label="cycle max")
ax.axhline(0.3, color="darkorange", ls=":", lw=1); ax.axhline(1.0, color="crimson", ls=":", lw=1)
ax.set_xlim(-6, 6); ax.set_xlabel("distance from outfall (km)")
ax.set_ylabel("excess temperature (K)"); ax.legend(fontsize=7, ncol=2)
ax.set_title("Far-field plume across the tidal cycle (S1)")

ax = fig.add_subplot(gs[0, 2])
m = r["t"] >= 2*T_M2
ax.plot(r["t"][m]/3600, r["series"]["outfall"][m], lw=1, label="outfall")
ax.plot(r["t"][m]/3600, r["series"]["intake"][m], lw=1, label=f"intake @{PLANT['x_intake']:.0f} m")
ax.plot(RES["A5_intake_too_close"]["_1d"]["t"][m]/3600,
        RES["A5_intake_too_close"]["_1d"]["series"]["intake"][m], lw=1, ls="--",
        label="intake @250 m (A5)")
ax.set_xlabel("hours"); ax.set_ylabel("theta (K)"); ax.legend(fontsize=7)
ax.set_title("Intake recirculation")

ax = fig.add_subplot(gs[1, 0])
r2 = RES["S1_discharge_only"]["_2d"]
pc = ax.pcolormesh(r2["x"]/1000, r2["y"]/1000, r2["theta_max"], shading="auto", cmap="inferno")
ax.contour(r2["x"]/1000, r2["y"]/1000, r2["theta_max"], levels=[0.3, 1.0],
           colors=["w", "cyan"], linewidths=.9)
plt.colorbar(pc, ax=ax, label="max theta (K)")
ax.set_xlabel("km"); ax.set_ylabel("km")
ax.set_title(f"S1 — mixing zone at u={RES['S1_discharge_only']['u_design_ms']:.2f} m/s", fontsize=8)

ax = fig.add_subplot(gs[1, 1])
r2 = RES["A1_slack_low_exchange"]["_2d"]
pc = ax.pcolormesh(r2["x"]/1000, r2["y"]/1000, r2["theta_max"], shading="auto", cmap="inferno")
ax.contour(r2["x"]/1000, r2["y"]/1000, r2["theta_max"], levels=[0.3, 1.0],
           colors=["w", "cyan"], linewidths=.9)
plt.colorbar(pc, ax=ax, label="max theta (K)")
ax.set_xlabel("km"); ax.set_ylabel("km")
ax.set_title(f"A1 — weak exchange, u={RES['A1_slack_low_exchange']['u_design_ms']:.2f} m/s", fontsize=8)

ax = fig.add_subplot(gs[1, 2])
sub = num.drop(index="S0_baseline_no_discharge")
xp = np.arange(len(sub))
ax.bar(xp-0.2, sub["dT_nf_K"], 0.4, label="near-field dT", color="#C0392B")
ax.bar(xp+0.2, sub["dT_far_max_K"], 0.4, label="far-field max dT", color="#1F3A5F")
ax.axhline(1.0, color="k", ls=":", lw=1)
ax.set_yscale("log"); ax.set_xticks(xp)
ax.set_xticklabels(sub.index, rotation=55, ha="right", fontsize=6.5)
ax.set_ylabel("dT (K)"); ax.legend(fontsize=7); ax.set_title("Near vs far field")
plt.show()

## Part 7 — Water-quality coupling: warming does not act alone

Heat is only one of the stressors it creates. Three couplings turn a modest ΔT into a
disproportionate biological load, and all three run in the *same* direction.

**1. Oxygen solubility falls.** Air-equilibrium O₂ follows Garcia & Gordon (1992), fitted to
Benson & Krause (1984):

$$\ln C_{O_2}^{*} = \sum_{i=0}^{5} A_i T_s^{i} + S\sum_{i=0}^{3}B_i T_s^{i} + C_0S^2,
\qquad T_s=\ln\!\frac{298.15-t}{273.15+t}$$

**2. Oxygen demand rises.** Ectotherm respiration scales with a $Q_{10}$ of roughly 2:

$$R(T) = R_{ref}\,Q_{10}^{(T-T_{ref})/10}$$

So the supply falls while the demand rises — the scissors that make thermal discharge
dangerous in water that was already near its oxygen limit. The Pacific Northwest shelf is
exactly such water: upwelling-driven nearshore hypoxia is a documented recurring feature.

**3. Ammonia becomes more toxic.** Only un-ionised NH₃ crosses the gill. Its fraction
follows Emerson et al. (1975) with a marine ionic-strength correction:

$$pK_a = 0.09018 + \frac{2729.92}{T+273.15},\qquad
f_{NH_3}=\frac{1}{1+10^{\,pK_a-\mathrm{pH}}}$$

$pK_a$ falls with temperature, so **the same total ammonia is roughly twice as toxic at
22 °C as at 10 °C** — which matters enormously for the contained-aquaculture idea in Part 9.

**Vetting:** the O₂ implementation is checked against TEOS-10 reference outputs, and the
ammonia implementation against the published Emerson table.

### Library note — DO and ammonia (Correction 5)

DO uses Garcia & Gordon (1992) via `ag.water_quality.do_saturation_mg_l` (cm³/dm³ coefficients → mg/L; verified in W-1/W-2). Ammonia uses Emerson et al. (1975) pKa with the Khoo seawater ionic-strength correction in the form used by Florida DEP (`ag.water_quality.unionised_fraction`). An earlier notebook form used a different ionic-strength algebra — seawater fractions may shift slightly. See [docs/CORRECTIONS.md](../docs/CORRECTIONS.md) item 5.

In [ ]:
# INTEGRATED: aquatic_guardian library
def o2_sat_mgL(T, S):
    """Dissolved O2 saturation (mg/L) via ag.water_quality.do_saturation_mg_l
    (Garcia & Gordon 1992, cm3/dm3 coefficients → mg/L)."""
    T = np.asarray(T, float); S = np.asarray(S, float)
    return np.vectorize(ag.water_quality.do_saturation_mg_l)(T, S)

def f_nh3_marine(T, pH, S=31.0):
    """Un-ionised NH3 fraction via ag.water_quality.unionised_fraction
    (Emerson et al. 1975 pKa + Khoo seawater ionic-strength correction — Correction 5)."""
    T = np.asarray(T, float)
    frac = np.vectorize(lambda t: ag.water_quality.unionised_fraction(float(t), float(pH), float(S)))(T)
    pka = np.vectorize(lambda t: ag.water_quality.pka_nh4(float(t), float(S)))(T)
    return frac, pka

def q10_rate(T, T_ref=10.0, Q10=2.0):
    return Q10**((np.asarray(T, float)-T_ref)/10.0)

# ---- VET O2 against library published check values (W-1 / W-2) --------------
assert abs(float(o2_sat_mgL(20, 0)) - 9.09) < 0.05
assert 7.2 <= float(o2_sat_mgL(20, 35)) <= 7.5
print(f"[VET] library DO sat 20 C fresh = {float(o2_sat_mgL(20,0)):.2f} mg/L "
      f"(~9.09); 20 C / 35 PSU = {float(o2_sat_mgL(20,35)):.2f} mg/L -> PASS")

# ---- VET ammonia against Emerson freshwater table --------------------------
_f_fw = ag.water_quality.unionised_fraction(20.0, 8.0, 0.0)
assert abs(100*_f_fw - 3.83) < 0.1, "Emerson NH3 implementation disagrees with the table"
print(f"[VET] Emerson NH3 at 20 C / pH 8.0 (fresh): {100*_f_fw:.2f} % vs published 3.83 % -> PASS")

Tb = SITE["T_amb_summer_7DADM"]
grid = pd.DataFrame({"dT_K": [0, 0.3, 1.0, 2.2, 4.0, 8.0]})
grid["T_C"]        = Tb + grid.dT_K
grid["O2_sat_mgL"] = [float(o2_sat_mgL(t, SITE["S_amb"])) for t in grid.T_C]
grid["O2_lost_%"]  = 100*(1 - grid.O2_sat_mgL/grid.O2_sat_mgL.iloc[0])
grid["resp_x"]     = [float(q10_rate(t, Tb, 2.0)) for t in grid.T_C]
grid["NH3_frac_%"] = [100*float(f_nh3_marine(t, SITE["pH_amb"], SITE["S_amb"])[0]) for t in grid.T_C]
grid["NH3_tox_x"]  = grid["NH3_frac_%"]/grid["NH3_frac_%"].iloc[0]
grid["O2_supply_per_demand"] = (grid.O2_sat_mgL/grid.O2_sat_mgL.iloc[0])/grid.resp_x
display(grid.round(3).style.background_gradient(
    subset=["O2_lost_%", "resp_x", "NH3_tox_x"], cmap="Reds")
    .set_caption(f"Compounding stressors relative to the summer baseline T = {Tb} C"))

fig, ax = plt.subplots(1, 3, figsize=(12.5, 3.2))
TT = np.linspace(6, 26, 200)
ax[0].plot(TT, o2_sat_mgL(TT, SITE["S_amb"]), color="#1F3A5F", label=f"S={SITE['S_amb']}")
ax[0].plot(TT, o2_sat_mgL(TT, 0), color="#7FA6C9", ls="--", label="S=0 (fresh)")
ax[0].axvline(Tb, color="k", ls=":", lw=.8)
ax[0].set_xlabel("T (C)"); ax[0].set_ylabel("O2 at air equilibrium (mg/L)")
ax[0].legend(fontsize=7); ax[0].set_title("Supply falls")
ax[1].plot(TT, q10_rate(TT, Tb, 2.0), color="#C0392B", label="Q10 = 2.0")
ax[1].plot(TT, q10_rate(TT, Tb, 2.5), color="#E8A0A0", ls="--", label="Q10 = 2.5")
ax[1].axvline(Tb, color="k", ls=":", lw=.8)
ax[1].set_xlabel("T (C)"); ax[1].set_ylabel("respiration relative to baseline")
ax[1].legend(fontsize=7); ax[1].set_title("Demand rises")
ax[2].plot(TT, [100*float(f_nh3_marine(t, 7.9, SITE["S_amb"])[0]) for t in TT], label="pH 7.9")
ax[2].plot(TT, [100*float(f_nh3_marine(t, 8.2, SITE["S_amb"])[0]) for t in TT], label="pH 8.2")
ax[2].set_xlabel("T (C)"); ax[2].set_ylabel("un-ionised NH3 (% of TAN)")
ax[2].legend(fontsize=7); ax[2].set_title("Toxic fraction rises")
plt.tight_layout(); plt.show()

print("The three panels point the same way. A +2.2 K excursion at the summer baseline costs")
print(f"{grid['O2_lost_%'].iloc[3]:.1f}% of dissolved-oxygen headroom while raising oxygen "
      f"demand {100*(grid.resp_x.iloc[3]-1):.0f}% and ammonia toxicity "
      f"{100*(grid.NH3_tox_x.iloc[3]-1):.0f}%.")


## Part 8 — Ecological dose: what the animal experiences, not what the gridpoint reads

### 8.1 The Eulerian fallacy

Regulatory screening usually reports ΔT at fixed points. But a juvenile salmon does not sit
at a gridpoint — it drifts, and the tide carries it **back and forth through the same plume**.
Its exposure is a path integral:

$$D = \int_{t_0}^{t_1} \phi\!\left(T_{amb}(t) + \theta\big(X_p(t),t\big)\right)\,dt,
\qquad \dot X_p = u(t) + w_{swim}$$

We therefore release Lagrangian particles and integrate exposure along their trajectories.
Particles released near slack water get re-advected through the plume on the return flood —
which produces exposures that no Eulerian point value reveals.

### 8.2 Three dose metrics, because "how hot" is not one question

| Metric | Form | Answers |
|---|---|---|
| **Thermal performance** | $\phi_{TPC}(T)$, beta/Yan–Hunt cardinal curve | how much capacity is lost *while* warm |
| **Aerobic scope** | Sharpe–Schoolfield with high-T inactivation | whether oxygen supply can meet demand |
| **Thermal dose** | $\displaystyle D=\sum \frac{\Delta t}{t_{50}(T)}$, $t_{50}=t_{ref}2^{-(T-T_{ref})/z}$ | whether cumulative exposure reaches a damage threshold |

Thresholds below are the published West Coast values, and they are **deliberately encoded as
data with a source string** so a reviewer can challenge each one individually.

In [ ]:
SPECIES = pd.DataFrame([
 # name, Tmin, Topt, Tmax(CTM), 7DADM criterion, source
 ("Chinook juvenile (rearing)",  4.0, 15.0, 25.1, 16.0,
  "EPA R10 core rearing 16 C; UILT 25.1 C (Brett 1952); HSP expression from 20 C"),
 ("Coho juvenile",               4.0, 14.5, 25.1, 16.0,
  "growth cessation > 20.3 C; C. shasta mortality ~100% at >= 20.6 C"),
 ("Sockeye (Fraser, Weaver Ck)", 4.0, 14.0, 21.0, 16.0,
  "CTmax 21 C — population-specific (Eliason et al. 2011, Science)"),
 ("Sockeye (Fraser, Chilko)",    4.0, 16.0, 25.0, 16.0,
  "CTmax 25 C — same species, 4 C spread across populations"),
 ("Chum / pink",                 3.0, 13.5, 23.7, 16.0,
  "lowest UILT of the Pacific salmon at 23.7 C"),
 ("Eelgrass Zostera marina",     5.0, 17.0, 30.0, 20.0,
  "optimum 11.5-26 C; inhibition > 20 C; net heterotrophy ~28.6 C; mortality >30 C for >5 d"),
 ("Bull kelp Nereocystis",       5.0, 12.0, 20.0, 16.0,
  "canopy collapse during the 2013-16 NE Pacific marine heatwave"),
], columns=["species", "T_min", "T_opt", "T_max", "crit_7DADM", "source"])
display(SPECIES)

def tpc_beta(T, Tmin, Topt, Tmax):
    '''Normalised cardinal thermal-performance curve (beta / Yan-Hunt form), 0-1.'''
    T = np.asarray(T, float); out = np.zeros_like(T)
    m = (T > Tmin) & (T < Tmax)
    e = (Tmax-Topt)/(Topt-Tmin)
    out[m] = ((T[m]-Tmin)*(Tmax-T[m])**e)/((Topt-Tmin)*(Tmax-Topt)**e)
    return np.clip(out, 0, 1)

def sharpe_schoolfield(T_C, E=0.55, E_h=4.2, T_h_C=21.0, T_ref_C=14.0):
    '''Aerobic-scope proxy: Boltzmann rise with high-temperature inactivation.'''
    k = 8.617333262e-5
    Tk = np.asarray(T_C, float)+273.15; Tr = T_ref_C+273.15; Th = T_h_C+273.15
    rise = np.exp((E/k)*(1/Tr - 1/Tk))
    inact = (1+np.exp((E_h/k)*(1/Th - 1/Tr)))/(1+np.exp((E_h/k)*(1/Th - 1/Tk)))
    return rise*inact

def thermal_dose(T_series, dt_h, T_ref=20.0, z=2.0, t_ref_h=168.0):
    '''Cumulative dose; 1.0 = one full reference exposure (a week at T_ref).'''
    T = np.asarray(T_series, float)
    t50 = t_ref_h*2.0**(-(T-T_ref)/z)
    return float(np.sum(np.where(T > T_ref, dt_h/np.maximum(t50, 1e-9), 0.0)))

# ---- VET the TPC shape ------------------------------------------------------
_sp = SPECIES.iloc[0]
_T = np.linspace(_sp.T_min-1, _sp.T_max+1, 4001)
_p = tpc_beta(_T, _sp.T_min, _sp.T_opt, _sp.T_max)
_peak = float(tpc_beta(np.array([_sp.T_opt]), _sp.T_min, _sp.T_opt, _sp.T_max)[0])
assert abs(_T[int(np.argmax(_p))] - _sp.T_opt) < 0.05, "TPC peak is not at T_opt"
assert _p[0] == 0.0 and _p[-1] == 0.0, "TPC is not zero outside [T_min, T_max]"
assert abs(_peak - 1.0) < 1e-12, "TPC is not normalised to 1 at T_opt"
assert _p.max() <= 1.0 + 1e-12, "TPC exceeds 1 somewhere"
print(f"[VET] TPC: exactly 1.000000 at T_opt={_sp.T_opt} C, peak on the sample grid at "
      f"{_T[int(np.argmax(_p))]:.2f} C (grid spacing {_T[1]-_T[0]:.4f} C), zero outside "
      f"[T_min, T_max], never exceeds 1 -> PASS")

fig, ax = plt.subplots(1, 2, figsize=(11.5, 3.2))
TT = np.linspace(2, 30, 400)
for _, s in SPECIES.iterrows():
    ax[0].plot(TT, tpc_beta(TT, s.T_min, s.T_opt, s.T_max), lw=1.1, label=s.species)
ax[0].axvline(SITE["T_amb_summer_7DADM"], color="k", ls=":", lw=1)
ax[0].annotate(f"summer 7-DADM {SITE['T_amb_summer_7DADM']} C",
               (SITE["T_amb_summer_7DADM"], 1.02), fontsize=6.5, ha="center")
ax[0].set_xlabel("T (C)"); ax[0].set_ylabel("relative performance")
ax[0].legend(fontsize=6); ax[0].set_title("Thermal performance curves")
ax[1].plot(TT, sharpe_schoolfield(TT), color="#C0392B", lw=1.4)
ax[1].axvline(SITE["T_amb_summer_7DADM"], color="k", ls=":", lw=1)
ax[1].set_xlabel("T (C)"); ax[1].set_ylabel("aerobic scope (relative)")
ax[1].set_title("Sharpe-Schoolfield aerobic-scope proxy")
plt.tight_layout(); plt.show()

sens = []
for _, s in SPECIES.iterrows():
    base = float(tpc_beta(SITE["T_amb_summer_7DADM"], s.T_min, s.T_opt, s.T_max))
    row = dict(species=s.species, perf_at_ambient=base)
    for d in [0.3, 1.0, 2.2]:
        p = float(tpc_beta(SITE["T_amb_summer_7DADM"]+d, s.T_min, s.T_opt, s.T_max))
        row[f"loss_at_+{d}K_%"] = 100*(1-p/max(base, 1e-9))
    row["margin_to_CTM_K"] = s.T_max - SITE["T_amb_summer_7DADM"]
    sens.append(row)
sens = pd.DataFrame(sens).set_index("species")
display(sens.round(2).style.background_gradient(
    subset=[c for c in sens.columns if c.startswith("loss")], cmap="Reds")
    .set_caption("Performance lost per kelvin of added heat, at the SUMMER baseline"))
print("The thermal safety margin, not the plume size, is what a +1 K increment spends.")

### 8.3 Lagrangian exposure — releasing particles into the plume

Each particle is released at a random position and tidal phase, drifts with
$u(t)$ plus a small random-walk term representing turbulent dispersion
($\sigma = \sqrt{2K\Delta t}$), and optionally swims. We integrate its temperature history
and score it with all three dose metrics. The distribution of outcomes across particles is
the ecologically meaningful answer; the mean alone hides the tail.

In [ ]:
def lagrangian_exposure(res_1d, n_part=600, days=3.0, dt=300.0, K=None, w_swim=0.0,
                        T_amb_series=None, rng=None, release_span=2000.0):
    '''Drift particles through the tidal plume and score their temperature history.'''
    rng = rng or np.random.default_rng(RUN["SEED"])
    x_grid, th_max = res_1d["x"], res_1d["theta_max"]
    snaps = res_1d["snaps"]; phases = sorted(snaps.keys())
    K = SITE["K_long"] if K is None else K
    nst = int(days*86400/dt)
    Xp = rng.uniform(-release_span, release_span, n_part)
    t0 = rng.uniform(0, T_M2, n_part)
    Tamb = SITE["T_amb_summer_7DADM"] if T_amb_series is None else T_amb_series
    hist = np.zeros((n_part, nst), dtype=np.float32)
    sig = np.sqrt(2*K*dt)
    for k in range(nst):
        t = t0 + k*dt
        u = tidal_current(t, SITE["U_amps"], SITE["U_phases"])
        Xp = Xp + (u + w_swim)*dt + rng.normal(0, sig, n_part)
        ph = phases[int(((t[0] % T_M2)/T_M2)*len(phases)) % len(phases)]
        hist[:, k] = np.interp(Xp, x_grid, snaps[ph], left=0.0, right=0.0)
    return dict(theta=hist, dt_h=dt/3600., T_amb=Tamb, X_end=Xp)

def score_exposure(exp, sp_row):
    th = exp["theta"].astype(float); Ta = exp["T_amb"]; dth = exp["dt_h"]
    Tp = Ta + th
    base = float(tpc_beta(Ta, sp_row.T_min, sp_row.T_opt, sp_row.T_max))
    perf = tpc_beta(Tp, sp_row.T_min, sp_row.T_opt, sp_row.T_max)
    perf_loss = 100*(1 - perf.mean(axis=1)/max(base, 1e-9))
    scope_base = float(sharpe_schoolfield(Ta))
    scope_loss = 100*(1 - sharpe_schoolfield(Tp).mean(axis=1)/scope_base)
    dose = np.array([thermal_dose(Tp[i], dth, T_ref=20.0, z=2.0) for i in range(Tp.shape[0])])
    return pd.DataFrame(dict(mean_dT=th.mean(axis=1), max_dT=th.max(axis=1),
                             hours_above_0p3=(th > 0.3).sum(axis=1)*dth,
                             perf_loss_pct=perf_loss, scope_loss_pct=scope_loss,
                             thermal_dose=dose))

sp = SPECIES.iloc[0]
EXP = {}
for nm in ["S1_discharge_only", "A1_slack_low_exchange", "A2_marine_heatwave"]:
    EXP[nm] = score_exposure(lagrangian_exposure(RES[nm]["_1d"], n_part=400 if RUN["FAST"] else 1200),
                             sp)

cmp = pd.DataFrame({k: v[["mean_dT", "max_dT", "hours_above_0p3",
                          "perf_loss_pct", "scope_loss_pct"]].mean() for k, v in EXP.items()}).T
cmp["p95_max_dT"] = [EXP[k].max_dT.quantile(.95) for k in cmp.index]
cmp["p95_perf_loss"] = [EXP[k].perf_loss_pct.quantile(.95) for k in cmp.index]
display(cmp.round(3).style.set_caption(
    f"Lagrangian exposure over 3 days — {sp.species} — mean across particles, plus the 95th percentile"))

fig, ax = plt.subplots(1, 3, figsize=(12.5, 3.2))
for nm, c in zip(EXP, ["#1F3A5F", "#C0392B", "#8E6C1F"]):
    ax[0].hist(EXP[nm].max_dT, bins=40, histtype="step", lw=1.3, color=c, label=nm)
    ax[1].hist(EXP[nm].hours_above_0p3, bins=40, histtype="step", lw=1.3, color=c)
    ax[2].hist(EXP[nm].perf_loss_pct, bins=40, histtype="step", lw=1.3, color=c)
ax[0].set_xlabel("max dT along the drift path (K)"); ax[0].legend(fontsize=6.5)
ax[0].set_ylabel("particles"); ax[0].set_title("Peak exposure")
ax[1].set_xlabel("hours above 0.3 K"); ax[1].set_title("Exposure duration")
ax[2].set_xlabel("mean performance loss (%)"); ax[2].set_title("Biological consequence")
plt.tight_layout(); plt.show()

e = EXP["S1_discharge_only"]
print(f"Eulerian view  : max far-field dT at a fixed point = {RES['S1_discharge_only']['dT_far_max_K']:.3f} K")
print(f"Lagrangian view: 95th-percentile particle sees     = {e.max_dT.quantile(.95):.3f} K")
print(f"                 and spends {e.hours_above_0p3.quantile(.95):.1f} h above 0.3 K over 3 days.")
print("\nThe tail matters: the tide re-advects particles through the plume, so a minority of")
print("drifters accumulate exposure far above the fixed-point average. Reporting only the")
print("Eulerian maximum systematically understates the ecological load.")

## Part 9 — Compliance screening

Now the modelled fields are tested against the criteria table from Part 1. Three distinct
kinds of test, because the criteria are of three distinct kinds:

- **ΔT criteria** (CCME ±1 K, EPA/WA/OR 0.3 K, CA 2.2 K) — tested at the near-field boundary
  and at the mixing-zone edge
- **Rate-of-change** (CCME 0.5 K/h) — only a time-resolved tidal model can produce this
- **Absolute 7-DADM** (13/16/17.5/20 °C) — tested as ambient + plume increment, which is why
  the summer baseline in `SITE` is the number that decides the outcome

### Library note — critical design conditions (Correction 4)

Mixing-zone compliance is evaluated at slack or low-percentile current, summer-maximum ambient, and maximum discharge — not at a steady mean current. `ag.criteria.temperature_rule` encodes WA marine and CA Thermal Plan increments. See [docs/CORRECTIONS.md](../docs/CORRECTIONS.md) item 4.

In [ ]:
# INTEGRATED: aquatic_guardian library
def sevenDADM(T_hourly, dt_h=1.0):
    per_day = max(1, int(round(24/dt_h))); nd = len(T_hourly)//per_day
    if nd < 1: return float(np.max(T_hourly))
    dmax = np.array([np.max(T_hourly[i*per_day:(i+1)*per_day]) for i in range(nd)])
    return float(dmax.mean()) if nd < 7 else float(np.max(np.convolve(dmax, np.ones(7)/7, "valid")))

def compliance_report(name, res, criteria=CRITERIA, binding=BINDING_SET,
                      T_base=None, mix_zone_m=100.0):
    T_base = SITE["T_amb_summer_7DADM"] if T_base is None else T_base
    r1 = res["_1d"]
    i_mz = int(np.argmin(np.abs(r1["x"]-mix_zone_m)))
    dT_mz = float(r1["theta_max"][i_mz])
    rows = []
    cr = criteria.set_index("id")
    for cid in binding:
        c = cr.loc[cid]
        if c["kind"] == "delta_T":
            for loc, val in [("near-field boundary", res["dT_nf_K"]),
                             (f"mixing-zone edge ({mix_zone_m:.0f} m)", dT_mz),
                             ("intake", res["dT_intake_K"])]:
                rows.append(dict(scenario=name, criterion=cid, location=loc,
                                 value=round(val, 4), limit=c["value"], unit=c["unit"],
                                 status="EXCEEDS" if val > c["value"] else "pass"))
        elif c["kind"] == "dT_dt":
            rows.append(dict(scenario=name, criterion=cid, location="250 m",
                             value=round(res["rate_max_K_per_h"], 4), limit=c["value"],
                             unit=c["unit"],
                             status="EXCEEDS" if res["rate_max_K_per_h"] > c["value"] else "pass"))
        elif c["kind"] == "abs_7DADM":
            T7 = T_base + dT_mz
            rows.append(dict(scenario=name, criterion=cid,
                             location=f"ambient {T_base} C + plume at {mix_zone_m:.0f} m",
                             value=round(T7, 3), limit=c["value"], unit=c["unit"],
                             status="EXCEEDS" if T7 > c["value"] else "pass"))
    return pd.DataFrame(rows)

COMP = pd.concat([compliance_report(k, v) for k, v in RES.items() if k != "S0_baseline_no_discharge"],
                 ignore_index=True)
piv = (COMP.assign(flag=(COMP.status == "EXCEEDS").astype(int))
           .pivot_table(index="scenario", columns="criterion", values="flag", aggfunc="max"))
display(piv.style.background_gradient(cmap="Reds", vmin=0, vmax=1)
        .set_caption("1 = at least one location exceeds the criterion"))
print("\nExceedances in detail:")
display(COMP[COMP.status == "EXCEEDS"].sort_values(["scenario", "criterion"]).head(25))

print("\nAbsolute-temperature screen (the one the ambient baseline decides):")
abs_tbl = []
for base in [11.5, 14.0, 16.8, 18.5]:
    row = dict(ambient_7DADM_C=base)
    for cid in ["EPA_R10_core", "EPA_R10_rear", "EPA_R10_migr"]:
        lim = float(CRITERIA.set_index("id").loc[cid, "value"])
        row[f"headroom_to_{cid}_K"] = round(lim-base, 2)
    abs_tbl.append(row)
display(pd.DataFrame(abs_tbl))
print("At a summer 7-DADM of 16.8 C the headroom to the core-rearing criterion is NEGATIVE.")
print("A +0.3 K increment is not small against a budget that is already overdrawn; this is the")
print("quantitative form of the 'thermal envelope is exhausted' finding from the governance work.")

# Critical design condition (Correction 4): library WA / CA rules at summer ambient
_Tcrit = SITE["T_amb_summer_7DADM"]
print("\n[library] temperature_rule at critical ambient", _Tcrit, "C:")
for _rule in ("wa_marine", "ca_new_coastal"):
    print(" ", _rule, ag.criteria.temperature_rule(_rule, _Tcrit))


### 9.1 Intake entrainment — the §316(b) side of the ledger

Thermal discharge is only half the regulatory exposure. A cooling-water intake kills
organisms by **impingement** (pinned against the screen) and **entrainment** (drawn through
the plant). The screening estimate is a volumetric one: the intake removes a fraction of
the water column's larval standing stock every day.

$$N_{entrained}\;=\;C_{larvae}\cdot q\cdot 86400\cdot(1-f_{survival})$$

This is the calculation that makes the $\Delta T_{dis}$ trade-off from Part 2 concrete — the
"gentle" low-ΔT design is the one that kills the most larvae.

In [ ]:
LARVAE = dict(conc_per_m3=2.5, survival=0.05, season_days=120,
              note="illustrative nearshore ichthyoplankton density; replace with site survey")

rows = []
for dT in [4, 6, 8, 10, 12, 15]:
    hb = dc_heat(PLANT["P_IT_MW"], PLANT["PUE"], PLANT["f_to_water"], dT,
                 SITE["T_amb_mean"], SITE["S_amb"])
    nf = near_field(hb["q_m3s"], dT, PLANT["n_ports"], PLANT["port_D"], SITE["depth_H"],
                    u_lo, T_amb=SITE["T_amb_mean"], S_amb=SITE["S_amb"])
    N = LARVAE["conc_per_m3"]*hb["q_m3s"]*86400*(1-LARVAE["survival"])
    rows.append(dict(dT_dis_K=dT, q_m3s=round(hb["q_m3s"], 2), q_MGD=round(hb["q_MGD"]),
                     dT_nf_K=round(nf["dT_nf"], 3),
                     larvae_per_day=f"{N:,.0f}",
                     larvae_per_season=f"{N*LARVAE['season_days']:,.0f}",
                     bta_316b="required" if hb["q_MGD"] > 2 else "n/a"))
ent = pd.DataFrame(rows)
display(ent.style.set_caption("Intake volume vs plume temperature: the design trade-off, costed both ways"))

fig, ax = plt.subplots(figsize=(6.4, 3.2))
d = np.array([r["dT_dis_K"] for r in rows])
ax.plot(d, [float(r["larvae_per_day"].replace(",", ""))/1e6 for r in rows], "o-",
        color="#1F3A5F", label="larvae entrained (millions/day)")
ax2 = ax.twinx()
ax2.plot(d, [r["dT_nf_K"] for r in rows], "s--", color="#C0392B",
         label="near-field dT (K)")
ax2.axhline(1.0, color="crimson", ls=":", lw=.8)
ax.set_xlabel("discharge dT (K)"); ax.set_ylabel("millions of larvae per day", color="#1F3A5F")
ax2.set_ylabel("near-field dT (K)", color="#C0392B")
ax.set_title("The two harms move in opposite directions")
fig.legend(loc="upper center", bbox_to_anchor=(0.55, 1.14), ncol=2, fontsize=7)
plt.tight_layout(); plt.show()
print("A design chosen to satisfy a thermal criterion will fail an entrainment criterion and")
print("vice versa. Screens, velocity caps and variable-speed pumps are the standard BTA")
print("responses; the SONGS mitigation programme is the precedent for what compensation at")
print("scale costs. Neither is modelled here — this is the flag, not the answer.")

## Part 10 — Beneficial use: is any of this heat actually worth anything?

### 10.1 The exergy problem

A 130 MW heat stream at 20 °C sounds enormous. It is not, thermodynamically. The fraction of
a heat stream convertible to work is the Carnot factor:

$$\eta_{ex} = 1 - \frac{T_0}{T_H}$$

At $T_H = 20$ °C and $T_0 = 10$ °C this is **3.4 %**. Data-centre waste heat sits in the
20–45 °C band — usable for *heating* but nearly worthless for anything requiring work, and
too cool to enter a conventional 60–90 °C district-heating network without a heat pump.
Reported COPs for real installations span roughly 2.2 (retrofit to an existing network) to
3.5–6 (low-temperature or "cold" networks needing only a small lift).

The three honest uses of low-grade heat, in descending order of thermodynamic sense:

1. **Direct use at source temperature** — no heat pump at all. A contained aquaculture loop
   or a greenhouse root zone can accept 20–30 °C water directly. This is the only option
   where the exergy penalty is zero.
2. **Small-lift heat pump** into a low-temperature (4th/5th-generation) network.
3. **Large-lift heat pump** into a legacy 70–90 °C network — usually where the economics die.

### 10.2 The structural finding this notebook keeps producing

Part 6 showed tidal dispersion drives open-water ΔT to hundredths of a kelvin within a
kilometre. **Heat that has been dispersed is gone.** So beneficial reuse is only viable
through **capture at the outfall, before discharge** — never by farming in warmed open water.
That is not a preference; it is what the transport model implies, and it also happens to be
the only version compatible with the charter's exclusion of releasing aquaculture organisms
into the wild.

In [ ]:
def carnot_cop(T_src_C, T_sink_C, eta_2nd=0.50):
    Th, Tc = T_sink_C+273.15, T_src_C+273.15
    return np.inf if Th <= Tc else eta_2nd*Th/(Th-Tc)

def exergy_fraction(T_src_C, T_dead_C=10.0):
    return max(0.0, 1 - (T_dead_C+273.15)/(T_src_C+273.15))

USES = pd.DataFrame([
 ("contained RAS aquaculture loop", 16, 22, "direct",       "fish tank inlet; species/life-stage limited"),
 ("greenhouse root-zone heating",   18, 25, "direct",       "hydroponic slab / soil warming"),
 ("greenhouse air heating",         35, 45, "heat pump",    "needs lift from 20-25 C source"),
 ("low-temperature (5GDH) network", 30, 40, "small lift",   "COP 3.5-6 reported"),
 ("legacy district heating",        70, 90, "large lift",   "COP ~2.2 reported; usually uneconomic"),
 ("shellfish / algae nursery",      14, 20, "direct",       "narrow window; biosecurity dominant"),
], columns=["use", "T_need_lo_C", "T_need_hi_C", "route", "note"])

T_src = SITE["T_amb_mean"] + PLANT["dT_dis"]
USES["COP_if_heat_pumped"] = [round(float(carnot_cop(T_src, t)), 2) for t in USES.T_need_hi_C]
USES["direct_feasible"] = USES.T_need_lo_C <= T_src
USES["exergy_of_source_%"] = round(100*exergy_fraction(T_src, SITE["T_amb_mean"]), 2)
display(USES.style.set_caption(f"Source stream at {T_src:.1f} C "
        f"(ambient {SITE['T_amb_mean']} C + {PLANT['dT_dis']} K)"))

cap = np.linspace(0, 0.9, 19)
rows = []
for f in cap:
    r = run_scenario(scenario_config(f_capture=f), do_2d=False, n_cycles=2)
    hb = dc_heat(PLANT["P_IT_MW"], PLANT["PUE"], PLANT["f_to_water"], PLANT["dT_dis"],
                 SITE["T_amb_mean"], SITE["S_amb"])
    rows.append(dict(f_capture=f, heat_captured_MW=hb["Q_water_MW"]*f,
                     dT_nf_K=r["dT_nf_K"], dT_far_max_K=r["dT_far_max_K"],
                     rate_K_per_h=r["rate_max_K_per_h"]))
capdf = pd.DataFrame(rows)

fig, ax = plt.subplots(1, 2, figsize=(11, 3.3))
ax[0].plot(capdf.f_capture*100, capdf.heat_captured_MW, color="#2E7D82", lw=1.6)
ax[0].set_xlabel("heat captured at the outfall (%)"); ax[0].set_ylabel("MW available for reuse")
ax[0].set_title("What capture buys you")
ax[1].plot(capdf.f_capture*100, capdf.dT_nf_K, color="#C0392B", lw=1.6, label="near-field dT")
ax[1].axhline(1.0, color="crimson", ls=":", label="CCME 1 K")
ax[1].axhline(0.3, color="darkorange", ls=":", label="0.3 K increment")
ax[1].set_xlabel("heat captured at the outfall (%)"); ax[1].set_ylabel("dT (K)")
ax[1].legend(fontsize=7); ax[1].set_title("What capture buys the ecosystem")
plt.tight_layout(); plt.show()

need = capdf[capdf.dT_nf_K <= 1.0]
print(f"Capture needed to bring the near-field boundary under CCME 1 K: "
      f"{100*need.f_capture.min():.0f}%" if len(need) else "not achievable by capture alone")
print(f"At that capture level, {need.heat_captured_MW.min() if len(need) else 0:.0f} MW is available")
print("as a 19.5 C stream — enough for direct-use applications, worth ~"
      f"{100*exergy_fraction(T_src, SITE['T_amb_mean']):.1f}% of its energy as exergy.")
print("\nNote the symmetry: capture is simultaneously the mitigation and the product. That is")
print("the only configuration in which 'beneficial reuse' is not a euphemism for 'discharge'.")

### 10.3 Agriculture: matching the heat stream to a real demand curve

A greenhouse is a **time-varying** heat sink, and the data centre is a **constant** heat
source. The mismatch, not the annual total, is what decides viability.

Greenhouse heating load is driven by the envelope conductance and the indoor-outdoor gap:

$$\dot Q_{gh}(t) = U A\,\big[T_{set} - T_{out}(t)\big]^{+} - \eta_{sol}\,A\,I(t)$$

Crop development is driven by **growing degree-days** above a base temperature:

$$\mathrm{GDD} = \sum_t \max\!\big(\bar T_{day}(t) - T_{base},\,0\big)$$

The comparison below is the actionable one: how much of the year's greenhouse demand can a
given capture fraction serve, how much heat is dumped because nobody needs it in August,
and how many extra crop cycles the warming buys.

In [ ]:
GH = dict(area_m2=20000, U_W_m2K=3.2, T_set_C=18.0, eta_solar=0.45,
          crop="tomato", T_base_C=10.0, GDD_to_harvest=1400)

doy = np.arange(365)
T_out = 9.5 + 7.5*np.sin(2*np.pi*(doy-110)/365) + 0.0
I_sol = 120 + 110*np.sin(2*np.pi*(doy-80)/365)
Q_gh = np.maximum(GH["U_W_m2K"]*GH["area_m2"]*(GH["T_set_C"]-T_out), 0) \
       - GH["eta_solar"]*GH["area_m2"]*I_sol
Q_gh = np.maximum(Q_gh, 0)/1e6                      # MW

hb = dc_heat(PLANT["P_IT_MW"], PLANT["PUE"], PLANT["f_to_water"], PLANT["dT_dis"],
             SITE["T_amb_mean"], SITE["S_amb"])
rows = []
for f in [0.1, 0.2, 0.4, 0.6, 0.8]:
    avail = hb["Q_water_MW"]*f
    served = np.minimum(Q_gh, avail)
    rows.append(dict(f_capture=f, available_MW=round(avail, 1),
                     demand_TWh_yr=round(Q_gh.sum()*24/1e6, 4),
                     demand_met_pct=round(100*served.sum()/max(Q_gh.sum(), 1e-9), 1),
                     surplus_dumped_pct=round(100*np.maximum(avail-Q_gh, 0).sum()
                                              / max(avail*365, 1e-9), 1),
                     days_fully_served=int((avail >= Q_gh).sum())))
gh = pd.DataFrame(rows)
display(gh.style.set_caption(f"{GH['area_m2']/1e4:.1f} ha greenhouse, U={GH['U_W_m2K']} W/m2K, "
                             f"setpoint {GH['T_set_C']} C"))

T_in_unheated = T_out + GH["eta_solar"]*I_sol/(GH["U_W_m2K"])
T_in_heated = np.maximum(T_in_unheated, GH["T_set_C"])
gdd_u = np.cumsum(np.maximum(T_in_unheated-GH["T_base_C"], 0))
gdd_h = np.cumsum(np.maximum(T_in_heated-GH["T_base_C"], 0))

fig, ax = plt.subplots(1, 3, figsize=(13, 3.2))
ax[0].plot(doy, Q_gh, color="#2E7D82", label="greenhouse demand")
for f, c in zip([0.2, 0.6], ["#C0392B", "#1F3A5F"]):
    ax[0].axhline(hb["Q_water_MW"]*f, ls="--", color=c, lw=1, label=f"capture {int(f*100)}%")
ax[0].set_xlabel("day of year"); ax[0].set_ylabel("MW"); ax[0].legend(fontsize=6.5)
ax[0].set_title("Constant source vs seasonal sink")
ax[1].plot(doy, gdd_u, label="unheated", color="#888")
ax[1].plot(doy, gdd_h, label="heated to setpoint", color="#C0392B")
ax[1].axhline(GH["GDD_to_harvest"], ls=":", color="k", label="harvest GDD")
ax[1].set_xlabel("day of year"); ax[1].set_ylabel("cumulative GDD")
ax[1].legend(fontsize=6.5); ax[1].set_title(f"{GH['crop']} development")
ax[2].bar(gh.f_capture*100, gh.demand_met_pct, width=8, color="#2E7D82", label="demand met")
ax[2].bar(gh.f_capture*100, gh.surplus_dumped_pct, width=8, bottom=0, alpha=.35,
          color="#C0392B", label="surplus dumped")
ax[2].set_xlabel("capture (%)"); ax[2].set_ylabel("% "); ax[2].legend(fontsize=6.5)
ax[2].set_title("Utilisation")
plt.tight_layout(); plt.show()

cyc_u = gdd_u[-1]/GH["GDD_to_harvest"]; cyc_h = gdd_h[-1]/GH["GDD_to_harvest"]
print(f"crop cycles per year: unheated {cyc_u:.2f} -> heated {cyc_h:.2f} "
      f"(+{100*(cyc_h/cyc_u-1):.0f}%)")
print("Real precedents for this configuration exist — the White Data Center on Hokkaido farms")
print("eels on ~33 C cooling water, and Equinix PA10 in Saint-Denis runs a rooftop hydroponic")
print("greenhouse off its cooling loop. Both are DIRECT-USE, land-side, contained systems.")
print("Neither warms open water, which is precisely the distinction this notebook defends.")

## Part 11 — Tidally-aware compute scheduling

### 11.1 The idea

The flushing capacity of the site is **not constant** — it swings with the tide by an order
of magnitude, and the compliance failures in Part 9 all occur near slack water. Meanwhile,
a large share of AI data-centre load (batch training, checkpointed fine-tuning, index builds,
render and ETL queues) is **deferrable by hours**.

So: shift deferrable compute out of slack water and into peak flow. The facility delivers
the same work per day while never emitting heat into water that cannot carry it away.

### 11.2 Formulation

Let $P_k$ be the IT power in hour $k$, split into a firm floor $P_{min}$ (inference, storage,
control plane) and a deferrable part. The instantaneous near-field temperature rise is
approximately proportional to the heat load and inversely to the available dilution:

$$\Delta T_{nf}(k) \;\approx\; \frac{\dot Q(P_k)}{\rho c_p\left[q + u(k)\,b\,H\right]}$$

Maximise delivered work subject to the criterion and to plant limits:

$$\max_{P}\ \sum_k P_k \quad\text{s.t.}\quad \Delta T_{nf}(k)\le \Delta T_{lim},\quad
P_{min}\le P_k \le P_{max},\quad \sum_k P_k \ge W_{required}$$

The per-hour constraint is monotone in $P_k$, so the optimum has a closed form — each hour
runs at the largest power its own dilution allows. Any deferred work is then re-placed into
the hours with remaining headroom, subject to a deadline. **No solver library is needed**,
which keeps this runnable on a free Colab runtime.

In [ ]:
def dilution_capacity(u, q, b_mix, H):
    '''Volumetric flux available to dilute the discharge in a current u (m3/s).'''
    return q + np.abs(u)*b_mix*H

def max_power_for_limit(u, dT_limit, dT_dis, PUE, f_to_water, b_mix, H, T_amb, S_amb,
                        P_cap_MW):
    '''Largest IT power whose near-field rise stays under dT_limit at current u.
    Q = rho cp q dT_dis and dT_nf = Q / (rho cp (q + u b H)) => solve for q, then for P.'''
    rho, cp = float(rho_sw(T_amb, S_amb)), float(cp_sw(T_amb, S_amb))
    ubH = np.abs(u)*b_mix*H
    denom = (dT_dis - dT_limit)
    q_max = np.where(denom > 1e-9, dT_limit*ubH/np.maximum(denom, 1e-9), np.inf)
    Q_max = rho*cp*q_max*dT_dis
    P_max = Q_max/(PUE*f_to_water*1e6)
    return np.clip(P_max, 0, P_cap_MW)

hours = np.arange(0, 24*14)
u_h = tidal_current(hours*3600., SITE["U_amps"], SITE["U_phases"])
b_mix = 3.0*SITE["depth_H"]
P_cap = PLANT["P_IT_MW"]
P_floor = 0.35*P_cap                       # firm inference / storage / control plane
dT_lim = 1.0                               # CCME

P_allow = max_power_for_limit(u_h, dT_lim, PLANT["dT_dis"], PLANT["PUE"],
                              PLANT["f_to_water"], b_mix, SITE["depth_H"],
                              SITE["T_amb_mean"], SITE["S_amb"], P_cap)

def schedule(P_allow, P_cap, P_floor, deferrable_frac=0.45, window_h=12):
    '''Greedy re-placement of deferrable work into hours with headroom, within a window.'''
    n = len(P_allow)
    firm = np.full(n, P_floor)
    headroom = np.maximum(np.minimum(P_allow, P_cap) - firm, 0.0)
    demand = np.full(n, (P_cap - P_floor)*deferrable_frac)
    placed = np.zeros(n); backlog = 0.0; missed = 0.0
    for k in range(n):
        want = demand[k] + backlog
        take = min(want, headroom[k])
        placed[k] = take
        backlog = want - take
        lo = max(0, k-window_h)
        if backlog > 0 and k >= window_h:
            expired = max(0.0, backlog - demand[lo:k+1].sum())
            missed += expired; backlog -= expired
    return dict(firm=firm, placed=placed, total=firm+placed, headroom=headroom,
                missed_MWh=missed, backlog=backlog)

sc = schedule(P_allow, P_cap, P_floor)
naive = np.full(len(hours), P_cap)
rho, cp = float(rho_sw(SITE["T_amb_mean"], SITE["S_amb"])), float(cp_sw(SITE["T_amb_mean"], SITE["S_amb"]))
def dT_of(P):
    q = P*PLANT["PUE"]*PLANT["f_to_water"]*1e6/(rho*cp*PLANT["dT_dis"])
    return q*PLANT["dT_dis"]/dilution_capacity(u_h, q, b_mix, SITE["depth_H"])

dT_naive, dT_sched = dT_of(naive), dT_of(sc["total"])

fig, ax = plt.subplots(3, 1, figsize=(11.5, 6.4), sharex=True)
ax[0].plot(hours/24, np.abs(u_h), color="#2E7D82", lw=.9)
ax[0].set_ylabel("|current| (m/s)"); ax[0].set_title("Tidal flushing capacity")
ax[1].fill_between(hours/24, 0, sc["firm"], color="#1F3A5F", alpha=.85, label="firm load")
ax[1].fill_between(hours/24, sc["firm"], sc["total"], color="#2E7D82", alpha=.8,
                   label="deferrable, tide-aware")
ax[1].plot(hours/24, np.minimum(P_allow, P_cap), color="crimson", lw=.8, ls="--",
           label="compliance ceiling")
ax[1].axhline(P_cap, color="k", lw=.6, ls=":", label="nameplate")
ax[1].set_ylabel("IT power (MW)"); ax[1].legend(fontsize=6.5, ncol=2)
ax[2].plot(hours/24, dT_naive, color="#C0392B", lw=.9, label="flat 100% load")
ax[2].plot(hours/24, dT_sched, color="#1F3A5F", lw=.9, label="tide-aware schedule")
ax[2].axhline(dT_lim, color="crimson", ls=":", label=f"limit {dT_lim} K")
ax[2].set_ylabel("near-field dT (K)"); ax[2].set_xlabel("days"); ax[2].legend(fontsize=6.5)
plt.tight_layout(); plt.show()

res = pd.DataFrame([
  dict(strategy="flat 100% load", MWh=naive.sum(),
       hours_over_limit=int((dT_naive > dT_lim).sum()),
       pct_time_over=round(100*np.mean(dT_naive > dT_lim), 1),
       peak_dT_K=round(float(dT_naive.max()), 3)),
  dict(strategy="tide-aware schedule", MWh=sc["total"].sum(),
       hours_over_limit=int((dT_sched > dT_lim).sum()),
       pct_time_over=round(100*np.mean(dT_sched > dT_lim), 1),
       peak_dT_K=round(float(dT_sched.max()), 3)),
]).set_index("strategy")
res["compute_retained_%"] = (100*res.MWh/res.MWh.iloc[0]).round(1)
display(res)
print(f"\nDeferrable work that missed its {12} h deadline: {sc['missed_MWh']:.0f} MWh "
      f"({100*sc['missed_MWh']/max(naive.sum(),1):.1f}% of nameplate energy)")
print("\nThis is the notebook's most directly actionable output. It is a scheduling policy, not")
print("an ecological claim: it says how to spend a fixed thermal budget, given that the budget")
print("itself must be set by regulators and aquatic scientists, not by this model.")
print("Operationally it needs: a tide forecast (DFO/NOAA harmonic prediction is deterministic")
print("and available years ahead), a workload classifier, and a human-approved ceiling.")

## Part 12 — Uncertainty: a single number is not a result

Every parameter above carries real uncertainty, and several are order-of-magnitude uncertain
(dispersion $K$ spans 1–50 m²/s in nearshore literature). We therefore propagate
distributions rather than point values, and rank parameters by how much of the output
variance each explains.

Two outputs are reported:

- **Exceedance probability** — $P(\Delta T_{nf} > \Delta T_{lim})$, the decision-relevant
  quantity, not the mean
- **Sensitivity ranking** — squared Spearman correlation as a variance-share proxy, plus a
  one-at-a-time tornado. Both are shown because they disagree when interactions matter, and
  that disagreement is itself information.

In [ ]:
PRIORS = {
  "K_long":     ("lognorm",  np.log(12.0), 0.85, "nearshore dispersion, literature 1-50 m2/s"),
  "K_surf":     ("norm",     32.0,          9.0, "surface exchange coeff, 10-100 W/m2/K"),
  "depth_H":    ("norm",     SITE["depth_H"], 2.0, "site bathymetry at the outfall"),
  "U_scale":    ("norm",     1.0,          0.20, "current amplitude vs the harmonic fit"),
  "dT_dis":     ("norm",     PLANT["dT_dis"], 1.2, "operational variation in discharge dT"),
  "PUE":        ("norm",     PLANT["PUE"], 0.06, "seasonal + load-dependent PUE"),
  "b_mix":      ("lognorm",  np.log(3.0*SITE["depth_H"]), 0.35, "near-field width, 2H-6H"),
  "T_amb":      ("norm",     SITE["T_amb_summer_7DADM"], 1.1, "summer baseline interannual"),
}

def sample_priors(n, rng):
    out = {}
    for k, (kind, a, b, _) in PRIORS.items():
        out[k] = rng.lognormal(a, b, n) if kind == "lognorm" else rng.normal(a, b, n)
    out["K_long"] = np.clip(out["K_long"], 1, 60)
    out["K_surf"] = np.clip(out["K_surf"], 8, 110)
    out["depth_H"] = np.clip(out["depth_H"], 5, 40)
    out["U_scale"] = np.clip(out["U_scale"], 0.2, 2.0)
    out["dT_dis"] = np.clip(out["dT_dis"], 2, 20)
    out["b_mix"] = np.clip(out["b_mix"], 10, 200)
    return pd.DataFrame(out)

def cheap_metrics(p):
    '''Analytic surrogate of the chain: near-field dilution + steady far-field scale.
    Vetted against the full solver below.'''
    hb = dc_heat(PLANT["P_IT_MW"], p.PUE, PLANT["f_to_water"], p.dT_dis, p.T_amb, SITE["S_amb"])
    amps = {k: v*p.U_scale for k, v in SITE["U_amps"].items()}
    u10 = np.percentile(np.abs(tidal_current(np.arange(0, 7*86400, 600.), amps,
                                             SITE["U_phases"])), 10)
    nf = near_field(hb["q_m3s"], p.dT_dis, PLANT["n_ports"], PLANT["port_D"], p.depth_H,
                    max(u10, 0.005), b_mix=p.b_mix, T_amb=p.T_amb, S_amb=SITE["S_amb"])
    lam = p.K_surf/(hb["rho"]*hb["cp"]*p.depth_H)
    u_rms = np.sqrt(np.mean(tidal_current(np.arange(0, 7*86400, 600.), amps,
                                          SITE["U_phases"])**2))
    # L_mix: distance over which surface loss removes the heat that dispersion has spread
    Lmix = np.sqrt(p.K_long/max(lam, 1e-12))
    # far-field scale: heat load spread across a plume of width 3H moving at u_rms,
    # damped over L_mix
    dT_far = (hb["Q_water_W"]/(hb["rho"]*hb["cp"]*p.depth_H*3*p.depth_H
                               * max(u_rms, .01)*np.sqrt(2*np.pi))
              * np.exp(-1.0))
    return nf["dT_nf"], dT_far, nf["S"], Lmix

N = 400 if RUN["FAST"] else 2000
P = sample_priors(N, RNG)
vals = np.array([cheap_metrics(P.iloc[i]) for i in range(N)])
P["dT_nf"], P["dT_far"], P["S_nf"], P["L_mix_m"] = vals.T

pex = {lim: float((P.dT_nf > lim).mean()) for lim in [0.3, 1.0, 2.2]}
print("Exceedance probability at the near-field boundary:")
for k, v in pex.items():
    print(f"  P(dT_nf > {k} K) = {100*v:5.1f} %")
print(f"\ndT_nf: median {P.dT_nf.median():.2f} K, 5-95 % [{P.dT_nf.quantile(.05):.2f}, "
      f"{P.dT_nf.quantile(.95):.2f}] K")

sens = []
for k in PRIORS:
    rho_s = stats.spearmanr(P[k], P.dT_nf).statistic
    sens.append(dict(parameter=k, spearman=rho_s, variance_share_pct=100*rho_s**2,
                     note=PRIORS[k][3]))
sens = pd.DataFrame(sens).sort_values("variance_share_pct", ascending=False)
sens["variance_share_pct"] = 100*sens.variance_share_pct/sens.variance_share_pct.sum()

tor = []
for k, (kind, a, b, _) in PRIORS.items():
    base = P.median(numeric_only=True)
    lo, hi = P[k].quantile(.1), P[k].quantile(.9)
    o = []
    for v in (lo, hi):
        row = base.copy(); row[k] = v
        o.append(cheap_metrics(row)[0])
    tor.append(dict(parameter=k, low=o[0], high=o[1], swing=abs(o[1]-o[0])))
tor = pd.DataFrame(tor).sort_values("swing")

fig, ax = plt.subplots(1, 3, figsize=(13.5, 3.6))
ax[0].hist(P.dT_nf, bins=50, color="#1F3A5F", alpha=.85)
for lim, c in [(0.3, "darkorange"), (1.0, "crimson")]:
    ax[0].axvline(lim, color=c, ls="--", label=f"{lim} K  P={100*pex[lim]:.0f}%")
ax[0].set_xlabel("near-field dT (K)"); ax[0].set_ylabel("draws"); ax[0].legend(fontsize=7)
ax[0].set_title(f"Monte Carlo, N={N}")
ax[1].barh(sens.parameter, sens.variance_share_pct, color="#2E7D82")
ax[1].set_xlabel("share of output variance (%)"); ax[1].set_title("Global sensitivity (rank-based)")
y = np.arange(len(tor)); med = float(np.median(P.dT_nf))
ax[2].barh(y, tor.high-tor.low, left=np.minimum(tor.low, tor.high), color="#C0392B", alpha=.8)
ax[2].axvline(med, color="k", lw=.8)
ax[2].set_yticks(y); ax[2].set_yticklabels(tor.parameter, fontsize=7)
ax[2].set_xlabel("dT_nf (K) across the 10-90% range"); ax[2].set_title("One-at-a-time tornado")
plt.tight_layout(); plt.show()
display(sens.round(3))

# ---- VET the surrogate's near-field branch against the full chain -----------
chk = []
for f in [0.0, 0.3, 0.6]:
    full = run_scenario(scenario_config(f_capture=f), do_2d=False, n_cycles=2)
    row = P.median(numeric_only=True).copy()
    row["PUE"] = PLANT["PUE"]; row["dT_dis"] = PLANT["dT_dis"]
    row["depth_H"] = SITE["depth_H"]; row["U_scale"] = 1.0
    row["b_mix"] = 3.0*SITE["depth_H"]; row["T_amb"] = SITE["T_amb_mean"]
    hb_full = dc_heat(PLANT["P_IT_MW"], PLANT["PUE"], PLANT["f_to_water"],
                      PLANT["dT_dis"], SITE["T_amb_mean"], SITE["S_amb"], f)
    sur = cheap_metrics(row)[0]*(1-f) if f > 0 else cheap_metrics(row)[0]
    chk.append(dict(f_capture=f, full_chain_dT_nf_K=round(full["dT_nf_K"], 4),
                    surrogate_dT_nf_K=round(float(sur), 4),
                    ratio=round(float(sur)/max(full["dT_nf_K"], 1e-9), 4)))
chk = pd.DataFrame(chk)
display(chk.style.set_caption("Surrogate vs full chain at the prior medians"))
print("\n[VET] The Monte Carlo uses an analytic surrogate for speed. Its near-field branch calls")
print("the SAME near_field() function as the full chain, so dT_nf agrees to within the")
print("capture scaling; only the far-field length scale is approximated. Re-run with")
print("FAST=False and swap cheap_metrics -> run_scenario for a full-chain uncertainty run")
print("(roughly 40x slower).")

## Part 13 — Machine-readable safety rules, monitoring design, and handoff

### 13.1 Why the rule engine has hysteresis and debouncing

A threshold comparison is not a control rule. A raw `if dT > 1.0: shutdown` will chatter
across the limit as the tide turns, producing dozens of spurious shutdowns a day and
training operators to ignore it. Real rules need:

- **Hysteresis** — trip high, clear lower, so noise cannot cycle the state
- **Debounce** — a condition must persist for $N$ consecutive samples before acting
- **Fail-safe defaults** — missing, stale or out-of-range sensor data escalates; it never
  reads as "safe". This is the charter's explicit requirement that *loss of connectivity,
  low confidence, sensor failure, or out-of-range conditions must not silently produce a
  safe decision.*
- **Human-in-the-loop for anything irreversible** — the engine recommends; it does not act.

In [ ]:
RULES = [
 dict(id="R1_thermal_exclusion", severity=5, action="curtail_to_floor_and_notify",
      trip=dict(metric="dT_nf_K", op=">", value=2.2), clear=dict(value=1.6),
      debounce_n=3, basis="CA Thermal Plan enclosed-bay limit; well above CCME"),
 dict(id="R2_ccme_dT", severity=4, action="curtail_deferrable_and_notify",
      trip=dict(metric="dT_nf_K", op=">", value=1.0), clear=dict(value=0.7),
      debounce_n=4, basis="CCME/BC marine +/-1 C"),
 dict(id="R3_rate_of_change", severity=4, action="hold_load_and_notify",
      trip=dict(metric="rate_K_per_h", op=">", value=0.5), clear=dict(value=0.3),
      debounce_n=2, basis="CCME max 0.5 C/h human-induced change"),
 dict(id="R4_do_floor", severity=5, action="stop_reuse_loop_and_notify",
      trip=dict(metric="DO_mgL", op="<", value=5.0), clear=dict(value=6.2),
      debounce_n=2, basis="sublethal hypoxia threshold; set per species with advice"),
 dict(id="R5_absolute_7DADM", severity=5, action="curtail_to_floor_and_escalate",
      trip=dict(metric="T_7DADM_C", op=">", value=16.0), clear=dict(value=15.4),
      debounce_n=1, basis="EPA R10 core juvenile rearing"),
 dict(id="R6_nh3_unionised", severity=4, action="stop_feeding_and_notify",
      trip=dict(metric="NH3_mgL", op=">", value=0.02), clear=dict(value=0.012),
      debounce_n=3, basis="common aquaculture caution level; species-specific"),
 dict(id="R7_recirculation", severity=3, action="review_intake_geometry",
      trip=dict(metric="recirc_index", op=">", value=0.25), clear=dict(value=0.15),
      debounce_n=6, basis="intake re-entrains its own discharge"),
 dict(id="R8_data_quality", severity=5, action="fail_safe_curtail_and_escalate",
      trip=dict(metric="data_age_s", op=">", value=900), clear=dict(value=300),
      debounce_n=1, basis="stale or missing data must never read as compliant"),
]

class RuleEngine:
    '''Stateful evaluator with hysteresis, debouncing and fail-safe defaults.'''
    def __init__(self, rules):
        self.rules = rules
        self.state = {r["id"]: dict(active=False, count=0) for r in rules}
    def _cmp(self, v, op, x):
        return v > x if op == ">" else v < x
    def step(self, obs):
        fired = []
        for r in self.rules:
            st = self.state[r["id"]]; m = r["trip"]["metric"]
            v = obs.get(m, None)
            if v is None or (isinstance(v, float) and not np.isfinite(v)):
                fired.append(dict(id=r["id"], action="fail_safe_curtail_and_escalate",
                                  severity=5, reason=f"{m} missing or non-finite"))
                st["active"] = True; st["count"] = r["debounce_n"]; continue
            trip = self._cmp(v, r["trip"]["op"], r["trip"]["value"])
            clear = not self._cmp(v, r["trip"]["op"], r["clear"]["value"])
            if not st["active"]:
                st["count"] = st["count"]+1 if trip else 0
                if st["count"] >= r["debounce_n"]:
                    st["active"] = True
                    fired.append(dict(id=r["id"], action=r["action"], severity=r["severity"],
                                      reason=f"{m}={v:.4g} {r['trip']['op']} {r['trip']['value']}"))
            else:
                if clear:
                    st["count"] = 0; st["active"] = False
                else:
                    fired.append(dict(id=r["id"], action=r["action"], severity=r["severity"],
                                      reason=f"{m}={v:.4g} still above clear "
                                             f"{r['clear']['value']}"))
        return sorted(fired, key=lambda d: -d["severity"])

eng = RuleEngine(RULES)
t_h = np.arange(0, 48, 0.25)
u_t = tidal_current(t_h*3600., SITE["U_amps"], SITE["U_phases"])
q_op = H0["q_m3s"]
dT_t = q_op*PLANT["dT_dis"]/dilution_capacity(u_t, q_op, b_mix, SITE["depth_H"])
rate_t = np.abs(np.gradient(dT_t, 0.25))
DO_t = np.array([float(o2_sat_mgL(SITE["T_amb_summer_7DADM"]+d, SITE["S_amb"]))*0.82
                 for d in dT_t])
log = []
for i, tt in enumerate(t_h):
    obs = dict(dT_nf_K=float(dT_t[i]), rate_K_per_h=float(rate_t[i]), DO_mgL=float(DO_t[i]),
               T_7DADM_C=SITE["T_amb_summer_7DADM"]+float(dT_t[i]),
               NH3_mgL=0.004, recirc_index=RES["S1_discharge_only"]["recirc_index"],
               data_age_s=60 if i != 90 else 1800)     # inject one stale-data event
    for f in eng.step(obs):
        log.append(dict(hour=round(tt, 2), **f))
LOG = pd.DataFrame(log)
print(f"{len(LOG)} rule activations over 48 h of simulated operation")
if len(LOG):
    display(LOG.groupby(["id", "action", "severity"]).size().rename("activations").reset_index())
    print("\nFirst activations:")
    display(LOG.drop_duplicates("id").head(10))

fig, ax = plt.subplots(2, 1, figsize=(11, 4.2), sharex=True)
ax[0].plot(t_h, dT_t, color="#1F3A5F", lw=1)
ax[0].axhline(1.0, color="crimson", ls="--", lw=.8, label="R2 trip (1.0 K)")
ax[0].axhline(0.7, color="crimson", ls=":", lw=.8, label="R2 clear (0.7 K)")
ax[0].fill_between(t_h, 0.7, 1.0, color="crimson", alpha=.08)
ax[0].set_ylabel("near-field dT (K)"); ax[0].legend(fontsize=7)
ax[0].set_title("Hysteresis band prevents chatter as the tide turns")
ax[1].plot(t_h, rate_t, color="#2E7D82", lw=1)
ax[1].axhline(0.5, color="crimson", ls="--", lw=.8, label="R3 trip (0.5 K/h)")
ax[1].set_ylabel("|dT/dt| (K/h)"); ax[1].set_xlabel("hours"); ax[1].legend(fontsize=7)
plt.tight_layout(); plt.show()
print("\nNote R8: the injected stale-data sample fires a severity-5 fail-safe even though every")
print("other metric was nominal. That behaviour is the requirement, not a bug.")

### 13.2 Monitoring design — where the model says to put the sensors

The model is also a **sensor-siting tool**. A station is worth installing where the signal it
would measure is (a) large enough to detect above instrument noise and (b) informative, i.e.
it discriminates between scenarios. We score candidate positions on both.

In [ ]:
SENSOR_NOISE_K = 0.02          # typical field thermistor, well-calibrated
cand = np.array([50, 100, 150, 250, 400, 600, 900, 1400, 2000, 3000, 4500])
rows = []
for xc in cand:
    vals = {}
    for nm in ["S1_discharge_only", "A1_slack_low_exchange", "A3_chiller_fault"]:
        r = RES[nm]["_1d"]
        vals[nm] = float(r["theta_max"][int(np.argmin(np.abs(r["x"]-xc)))])
    signal = vals["S1_discharge_only"]
    discrim = max(vals.values()) - min(vals.values())
    rows.append(dict(x_m=xc, signal_K=signal, snr=signal/SENSOR_NOISE_K,
                     discrimination_K=discrim, detectable=signal > 3*SENSOR_NOISE_K,
                     inside_L_ex=xc < L_ex))
sensors = pd.DataFrame(rows)
sensors["priority"] = (sensors.snr.rank(pct=True) + sensors.discrimination_K.rank(pct=True)).round(2)
display(sensors.sort_values("priority", ascending=False).round(4)
        .style.set_caption("Candidate monitoring positions, ranked"))

best = sensors.sort_values("priority", ascending=False).head(3)
print("Recommended array (to be confirmed by the graduate student against site access):")
for _, s in best.iterrows():
    print(f"  {s.x_m:>6.0f} m  SNR {s.snr:6.1f}  discrimination {s.discrimination_K:.3f} K"
          f"  {'(inside tidal excursion)' if s.inside_L_ex else ''}")
print(f"\nPlus one REFERENCE station beyond {1.5*L_ex/1000:.1f} km (outside the excursion) so the")
print("ambient baseline is measured, not assumed. Without it no dT criterion can be evaluated,")
print("because every criterion here is a DIFFERENCE from a natural condition.")
print(f"Sampling interval must resolve the 0.5 K/h rate criterion: <= 15 min.")

### 13.3 Assumption and uncertainty register

Every number that could move a decision, with its provenance and how sensitive the result is
to it. A reviewer should be able to attack this table directly.

In [ ]:
REGISTER = pd.DataFrame([
 ("EOS-80 seawater properties", "UNESCO 1983", "verified vs published check values", "low", "verified"),
 ("O2 solubility", "Garcia & Gordon 1992 / Benson & Krause 1984", "verified vs TEOS-10", "low", "verified"),
 ("NH3 speciation", "Emerson et al. 1975 + Khoo ionic strength", "verified vs published table", "low", "verified"),
 ("advection-diffusion solver", "Strang split, semi-Lagrangian + Crank-Nicolson",
  "2nd-order convergence, analytic benchmarks, exact conservation", "low", "verified"),
 ("near-field jet/plume constants", "classical asymptotes (Fischer; Roberts; Jirka)",
  "empirical fits; invalid below F~5", "HIGH", "bounded by conservation ceiling"),
 ("near-field width b_mix", "assumed 3H", "no measurement", "HIGH", "in Monte Carlo"),
 ("longitudinal dispersion K", "literature 1-50 m2/s nearshore", "not measured at site", "HIGH", "in Monte Carlo"),
 ("surface exchange K_surf", "Edinger linearisation, 10-100 W/m2/K", "weather dependent", "medium", "in Monte Carlo"),
 ("tidal constituents", "SET_ME station record", "harmonic fit; no residual currents", "medium", "U_scale prior"),
 ("stratification", "NOT MODELLED - depth-averaged", "a buoyant plume can trap at the surface", "HIGH", "known gap"),
 ("wind- and density-driven residual flow", "NOT MODELLED", "can dominate at slack tide", "HIGH", "known gap"),
 ("bathymetry / shoreline", "NOT MODELLED - open domain", "embayments concentrate heat", "HIGH", "known gap"),
 ("thermal thresholds", "EPA R10 / WAC / OAR / CCME / primary literature", "regional, not site-specific", "medium", "documented per row"),
 ("TPC cardinal temperatures", "literature per species", "acclimation state not represented", "HIGH", "sensitivity shown"),
 ("larval concentration", "illustrative 2.5 /m3", "no site survey", "HIGH", "flagged, not used for decisions"),
 ("greenhouse envelope U", "assumed 3.2 W/m2K", "design dependent", "medium", "scenario input"),
 ("deferrable compute fraction", "assumed 45%", "facility specific", "medium", "scenario input"),
], columns=["assumption", "source", "note", "sensitivity", "treatment"])
display(REGISTER.style.apply(
    lambda s: ["background-color:#ffe0e0" if v == "HIGH" else
               ("background-color:#fff6d6" if v == "medium" else "background-color:#e6f6e6")
               for v in s] if s.name == "sensitivity" else ["" for _ in s]))
print(f"\n{(REGISTER.sensitivity=='HIGH').sum()} HIGH-sensitivity assumptions. Four of them are")
print("structural gaps (stratification, residual flow, bathymetry, acclimation) that no amount")
print("of tuning fixes -- they need a 3-D model and field data. State this in the report.")

In [ ]:
# INTEGRATED: aquatic_guardian library
# ============================ HANDOFF ARTIFACTS ==============================
import os
RUN["EXPORT_DIR"] = str(HANDOFF_DIR)
os.makedirs(RUN["EXPORT_DIR"], exist_ok=True)

num.to_csv(f"{RUN['EXPORT_DIR']}/scenario_summary.csv")
COMP.to_csv(f"{RUN['EXPORT_DIR']}/compliance_report.csv", index=False)
REGISTER.to_csv(f"{RUN['EXPORT_DIR']}/assumption_register.csv", index=False)
sens.to_csv(f"{RUN['EXPORT_DIR']}/sensitivity_ranking.csv", index=False)
sensors.to_csv(f"{RUN['EXPORT_DIR']}/monitoring_design.csv", index=False)
P.describe().to_csv(f"{RUN['EXPORT_DIR']}/uncertainty_summary.csv")

_T_loop = float(SITE["T_amb_mean"] + PLANT["dT_dis"] * (1 - PLANT.get("f_capture", 0.0)))
_nf1 = RES["S1_discharge_only"]
_dx = 25.0
_u_audit = float(U_eff)
_K_num = ag.numerics.k_num_upwind(_u_audit, _dx, None)
_K_phys = ag.numerics.okubo_kh(1000.0)
_placeholders = [r for r in REGISTER[REGISTER.sensitivity == "HIGH"].assumption.tolist()]
_placeholders = list(_placeholders) + ["kelp and eelgrass thresholds"]

rec = ag.handoff.make(
    "thermal_tidal",
    scenario_id="august-p10-current",
    assumptions=[
        "slack-water / p10-current near field",
        f"far-field semi-Lagrange+CN, dx~25 m; library K_num/K_phys reported",
        f"tidal regime {classify_regime(F)}",
    ],
    placeholders=_placeholders,
    loop_inlet_temperature_c=_T_loop,
    receiving_water={
        "ambient_t_c": float(SITE["T_amb_mean"]),
        "salinity_psu": float(SITE["S_amb"]),
        "ph": float(SITE["pH_amb"]),
        "do_mg_l": float(ag.water_quality.do_saturation_mg_l(SITE["T_amb_mean"], SITE["S_amb"])),
        "unionised_ammonia_fraction": float(
            ag.water_quality.unionised_fraction(SITE["T_amb_mean"], SITE["pH_amb"], SITE["S_amb"])),
    },
    plume={
        "delta_t_mixing_zone_edge_k": float(_nf1["dT_nf_K"]),
        "regime": "transitional",
        "near_field_dilution": float(max(PLANT["dT_dis"] / max(_nf1["dT_nf_K"], 1e-9), 1.0)),
        "design_condition": "10th-percentile current, summer ambient",
        "grid": {"dx_m": _dx, "k_num_m2s": _K_num, "k_phys_m2s": _K_phys, "scheme": "tvd"},
    },
)
ag.handoff.write(rec, HANDOFF_DIR / "thermal_handoff.json")
# Optional Colab: ag.handoff.write(rec, "/content/drive/MyDrive/aquatic_guardian/thermal_handoff.json")

# Keep a richer sidecar for teaching / debugging (not schema-validated)
HANDOFF = dict(schema_record=rec, safety_contract=SAFETY_CONTRACT,
               key_results=dict(
                   tidal_regime=classify_regime(F), form_factor=float(F), L_excursion_m=float(L_ex),
                   heat_to_water_MW=float(H0["Q_water_MW"]), intake_MGD=float(H0["q_MGD"]),
                   dT_nearfield_K=float(_nf1["dT_nf_K"]),
                   dT_farfield_max_K=float(_nf1["dT_far_max_K"]),
               ),
               aquaculture_handoff=dict(
                   loop_inlet_T_C=_T_loop,
                   ambient_T_C=float(SITE["T_amb_mean"]),
                   salinity=float(SITE["S_amb"]), pH=float(SITE["pH_amb"]),
                   DO_at_loop_T_mgL=float(o2_sat_mgL(_T_loop, SITE["S_amb"])),
                   NH3_fraction_at_loop_T=float(f_nh3_marine(_T_loop, SITE["pH_amb"], SITE["S_amb"])[0]),
               ))
with open(HANDOFF_DIR / "thermal_handoff_sidecar.json", "w") as f:
    json.dump(HANDOFF, f, indent=2, default=str)

print("Exported validated handoff:", HANDOFF_DIR / "thermal_handoff.json")
print("library_version:", rec["library_version"], "placeholders:", len(rec["placeholders"]))
_back = None
_back_path = HANDOFF_DIR / "aquaculture_handoff.json"
if _back_path.exists():
    _back = ag.handoff.read(_back_path)
    print("Aquaculture ceiling returned:", _back.get("recommended_loop_inlet_ceiling_c"),
          "binding:", _back.get("binding_constraint"))


### 13.4 Conditional finding — the template to complete

Fill this in from your own run. Every blank must be a number your notebook produced, and
every claim must survive the verification suite.

> For the modelled regime **{tidal regime}** with form factor **{F}** and tidal excursion
> **{L_ex} m**, a **{P_IT} MW** facility rejecting **{Q} MW** to seawater produces a
> near-field boundary excess of **{dT_nf} K** and a far-field maximum of **{dT_far} K**,
> decaying to **{dT_1km} K** at one kilometre. Recirculation at the assumed intake is
> **{recirc}**. Monte Carlo over the documented priors gives
> **P(ΔT > 1 K) = {P1}** and **P(ΔT > 0.3 K) = {P03}**.
>
> Because dispersion removes the heat from open water within **{L} m**, no recoverable
> thermal resource exists in the receiving environment. Beneficial reuse is therefore
> **viable only through capture at the outfall**, at a capture fraction of at least
> **{f_cap}%**, delivering **{MW_cap} MW** at **{T_src} °C** — a stream with an exergy
> content of **{eta}%**, suitable for direct-use contained aquaculture or greenhouse
> root-zone heating, and not for anything requiring work.
>
> **No-go conditions:** {list the rule IDs that fired, and the scenarios that triggered them}.
>
> **This is a screening result.** It has not been validated against field data or a 3-D
> hydrodynamic model, it does not represent stratification, residual currents or shoreline
> geometry, and it does not establish that any scenario is ecologically safe.

### 13.5 What is explicitly out of scope

3-D hydrodynamics (Delft3D / MOHID / ROMS), validated near-field mixing (CORMIX / CorJet),
biological response experiments, population-level modelling, autonomous control of cooling
or discharge, and legal or regulatory certification. Also unmodelled and flagged in the
governance work as unquantified for marine data centres: **electromagnetic fields** from
power cables, **continuous low-frequency noise** overlapping Southern Resident killer whale
communication bands, **antifouling and sacrificial-anode chemical leaching**, and
**biofouling/invasive-species** vectors. A thermal model is not an environmental assessment.

### 13.6 Repository checklist

- [ ] this notebook, run top to bottom on a fresh runtime, all `[VET]` lines PASS
- [ ] `SITE` completed with a real station id, retrieval date and source URL
- [ ] `PLANT` completed with the facility configuration you are actually screening
- [ ] the seven exported artifacts committed
- [ ] `thermal_handoff.json` passed to the aquaculture-vision notebook
- [ ] assumption register reviewed line by line with the supervisor
- [ ] every HIGH-sensitivity row either narrowed with data or named in the report as a gap
- [ ] the conditional finding completed, with numbers, and **reviewed by an aquatic scientist
      before it appears in any document that leaves the team**